# Laya V5.3 — Liquid CeNN FFN Experiment (Replacement vs Residual)

[![Open in Google Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V53_Liquid_CeNN_Colab.ipynb)

**Goal:** test a real **1D CeNN-inspired liquid recurrent layer** in the existing 28-layer compressed ModernBERT encoder. The CeNN operates across neighboring token positions using a learned depthwise 3-tap stencil and input-dependent relaxation gates:

`h_(k+1) = h_k + dt(x,h_k) * (tanh(W_x x + K * h_k + R h_k) - h_k)`

This is a **CeNN-inspired 1D token lattice**; it is not the canonical continuous-time Chua CeNN or a learned ODE solver. It can run in two modes:

1. **replace:** completely remove one original FFN output projection (`mlp.Wo`) and use only the liquid-CeNN cell. This can save projection parameters, but pretrained quality may suffer. Use a real-activation matching phase before evaluating it.
2. **residual:** retain the pretrained projection and add `scale * LiquidCeNN(x)`. `scale = 0` at initialization, giving **exact baseline output** and a safe trainable correction; this does not save memory and may slow inference.

## Experimental method

- Rebuild the exact 20 selectively compressed attention/FFN matrices from V5.1/V5.2 using deterministic SVD seeds, with the 28-layer pretrained ModernBERT encoder and original decision head unchanged.
- Optionally import your prior **V5.2 standalone checkpoint directory** to use the actually gold-tuned head as the baseline. Otherwise test against the reconstructed compressed model (not the unavailable V5.2 tuned model).
- Screen candidate FFN output layers **8, 17, 20, 23, 24, 26**; optional scan all 28. Capture representative real activations, train a small replacement cell to imitate the original projection output (MSE), then measure teacher agreement and gold accuracy on a **small gold-validation subset**. Do not use the TEST split for selection.
- Prefer a replacement **only** if it passes the acceptance screen. Otherwise choose a residual cell at the best-reconstructed layer and clearly label the result as **augmentation, not replacement**.
- Train the selected CeNN module on gold labels with a small teacher-KL regularizer and an optional final scoring-head fine-tune; select checkpoint on validation gold accuracy and divergence.
- Evaluate once on the dataset's separate TEST split; compare to unchanged baseline and teacher; report model bytes, GPU/wall latency, and teacher mistakes corrected or newly introduced.
- Export a strict offline **full FP32 standalone** checkpoint with its own liquid-CeNN loader. Publish to private `vtava/Laya-v53-liquid-cenn` **only if** all measured release gates pass.
- Final cell packages **all experiment reports, plots (if any), checkpoints and complete model** into a ZIP64 archive, with SHA-256 manifest and automatic Colab download. The ZIP happens even if HF publication is blocked.

**Known previous result:** V5.2 did not beat its teacher on gold answers (75.5% student vs 76.5% teacher across 200 test decisions). That result motivates a controlled architecture experiment but does not prove that CeNN dynamics will help.

[Liquid CeNN implementation](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/standalone/laya_v53_liquid_cenn/modeling_laya_v53.py) · [V5.2 control notebook](https://github.com/vtavakkoli/TinyCeNN-LM/blob/main/notebooks/Laya_V52_Gold_Aware_Supervised_Colab.ipynb)


In [1]:
#@title 1. Pinned requirements, GPU environment and runtime imports
%pip -q install "transformers==4.57.6" "datasets>=2.20" "huggingface_hub>=0.25,<1" "safetensors>=0.4"
import json,math,time,random,hashlib,copy,statistics,gc,importlib.util
from pathlib import Path
from collections import OrderedDict,Counter
import numpy as np
import torch
from torch import nn
from torch.nn import functional as F
from datasets import load_dataset
from huggingface_hub import HfApi,snapshot_download,notebook_login
from safetensors.torch import save_file,load_file
from transformers import AutoTokenizer
assert torch.cuda.is_available(),"Select a Colab GPU (T4 or better)"
DEVICE=torch.device("cuda")
print("Torch:",torch.__version__,"GPU:",torch.cuda.get_device_name(0))


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.0/44.0 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.0/12.0 MB 127.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 47.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.3/3.3 MB 120.6 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
diffusers 0.40.0 requires huggingface-hub<2.0,>=1.23.0, but you have huggingface-hub 0.36.2 which is incompatible.
gradio 6.29.0 requires huggingface-hub<3.0,>=1.16.0, but you have huggingface-hub 0.36.2 which is incompatible.
Torch: 2.11.0+cu130 GPU: Tesla T4


In [2]:
#@title 2. V5.3 config, pinned teacher, optional prior checkpoint and GPU
TEACHER_REPO="vtava/Laya-1.2-speedup-standalone"
PINNED_TEACHER_REVISION="b60184b2976e9f7ffa60c1c42b221d81b3691c3d"
STUDENT_REPO="vtava/Laya-v53-liquid-cenn"
SEED=2047
MAX_LEN=512
TRAIN_CASES=500
DEV_CASES=240
BATCH=2 #@param {type:"integer"}
TRAIN_B=1 #@param {type:"integer"}
TRAIN_MAX_TOKENS=224 #@param {type:"integer"}
CANDIDATE_LAYERS=(8,17,20,23,24,26)
SCAN_ALL_28_LAYERS=False #@param {type:"boolean"}
CE_WIDTH=64 #@param {type:"integer"}
CE_STEPS=3 #@param {type:"integer"}
CE_DT_BOUND=0.5
CAPTURE_BATCHES=5 #@param {type:"integer"}
IMITATION_STEPS=70 #@param {type:"integer"}
IMITATION_LR=0.001 #@param {type:"number"}
SCREEN_VAL_DECISIONS=48
GOLD_VAL_DECISIONS=180
HISTORICAL_PROBE_DECISIONS=120
AUDIT_DECISIONS=200
TRAIN_STEPS=200 #@param {type:"integer"}
TRAIN_CHECK_EVERY=20 #@param {type:"integer"}
TRAIN_LR=0.0001 #@param {type:"number"}
TRAIN_DECISION_SCORER=False #@param {type:"boolean"}
TEACHER_KL_WEIGHT=0.10
GOLD_CE_WEIGHT=0.85
GOLD_SOFT_WEIGHT=0.05
BASELINE_V52_STANDALONE_DIR="" #@param {type:"string"}
BENCH_REPEATS=10 #@param {type:"integer"}
MIN_VAL_GOLD_GAIN_PP=1.0
MIN_TEST_GOLD_GAIN_PP=1.0
MIN_TEST_BASELINE_GAIN_PP=0.0
MAX_TEST_JS=0.08
MIN_TEACHER_AGREEMENT=0.80
MIN_SPEED_RATIO=0.95
REQUIRE_SPEED_NONREGRESSION=False #@param {type:"boolean"}
MIN_CHECKPOINT_BYTES_SAVED=0
PUBLISH_TO_HF=True #@param {type:"boolean"}
PRIVATE_HF_REPO=True #@param {type:"boolean"}
ALLOW_UPDATE_EXISTING=False #@param {type:"boolean"}
assert 1<=TRAIN_B<=2 and 1<=BATCH<=8
assert 4<=CE_WIDTH<=512 and 1<=CE_STEPS<=10
assert 1<=CAPTURE_BATCHES<=32 and IMITATION_STEPS>=0
assert TRAIN_STEPS>=0 and TRAIN_CHECK_EVERY>=1 and BENCH_REPEATS>=5
DEVICE=torch.device("cuda")
device=DEVICE
amp_dtype=(torch.bfloat16 if torch.cuda.is_bf16_supported(including_emulation=False)
           else torch.float16)
random.seed(SEED);np.random.seed(SEED);torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
OUT=Path("/content/Laya_V53_LiquidCeNN")
OUT.mkdir(parents=True,exist_ok=True)
DEST=OUT/"standalone_model"
DEST.mkdir(parents=True,exist_ok=True)
api=HfApi()
try:
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
except Exception:
    notebook_login()
    teacher_info=api.model_info(TEACHER_REPO,revision=PINNED_TEACHER_REVISION)
assert teacher_info.sha==PINNED_TEACHER_REVISION
parent_dir=Path(snapshot_download(TEACHER_REPO,revision=PINNED_TEACHER_REVISION))
source_config=json.loads((parent_dir/"config.json").read_text())
assert source_config["format"]=="laya-1p2-full-standalone-v1"
assert source_config["head_context_budget"]==64
source_cfg={"head_max_len":192,"max_len":MAX_LEN}
tokenizer=AutoTokenizer.from_pretrained(str(parent_dir/"tokenizer"),local_files_only=True)
print("Original pretrained teacher:",TEACHER_REPO,teacher_info.sha)


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


Fetching 13 files:   0%|          | 0/13 [00:00<?, ?it/s]

README.md: 0.00B [00:00, ?B/s]

.gitattributes: 0.00B [00:00, ?B/s]

modeling_laya_standalone.py: 0.00B [00:00, ?B/s]

...

requirements.txt:   0%|          | 0.00/74.0 [00:00<?, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

Original pretrained teacher: vtava/Laya-1.2-speedup-standalone b60184b2976e9f7ffa60c1c42b221d81b3691c3d


In [3]:
#@title 3. Exact V5.3 standalone runtime, teacher and compressed student
V53_RUNTIME_CODE = "\"\"\"Laya V5.0 all-28-layer brute-force hybrid SVD compression, fully standalone.\n\nRequires only torch, transformers, safetensors. Does NOT download or import\nconvaiinnovations/laya-typed-decisions or TinyCeNN-LM at inference.\nThe model weights and tokenizer must be present in the same export directory.\nOption-score model only; auxiliary action logits are intentionally unavailable.\n\"\"\"\nfrom __future__ import annotations\nimport json\nfrom pathlib import Path\nfrom typing import Any, Mapping, Sequence\nimport torch\nfrom torch import Tensor, nn\nfrom torch.nn import functional as F\nfrom safetensors.torch import load_file\nfrom transformers import AutoModel, AutoConfig, AutoTokenizer\n\nQTYPES = {\"choice\": 0, \"score\": 1, \"noul\": 2}\n_DEFAULT_NOUL_LABELS = {\"false\": \"false\", \"true\": \"true\"}\n\n\ndef _serialize_state(state: str | dict | list) -> str:\n    if isinstance(state, str):\n        return state\n    return json.dumps(state, ensure_ascii=False, separators=(\",\", \":\"), default=str)\n\n\ndef _render_value(value: Any) -> str:\n    if isinstance(value, str):\n        return value\n    return json.dumps(value, ensure_ascii=False, separators=(\", \", \": \"), default=str)\n\n\ndef normalize_question(question: Mapping[str, Any]) -> dict[str, Any]:\n    \"\"\"Normalize an external typed-decision question into the runtime format.\"\"\"\n    qtype = str(question.get(\"type\", question.get(\"t\", \"choice\"))).lower()\n    if qtype not in QTYPES:\n        raise ValueError(f\"unsupported question type: {qtype!r}\")\n\n    criteria = question.get(\"criteria\", question.get(\"crit\"))\n    if qtype == \"choice\":\n        if isinstance(criteria, (list, tuple)):\n            criteria = {str(x): None for x in criteria}\n        if not isinstance(criteria, Mapping) or len(criteria) < 2:\n            raise ValueError(\"choice questions require at least two criteria/options\")\n        criteria = dict(criteria)\n    elif qtype == \"score\":\n        if not isinstance(criteria, (list, tuple)) or len(criteria) < 2:\n            raise ValueError(\"score questions require at least two ordered criteria\")\n        criteria = list(criteria)\n    else:\n        if criteria is None:\n            criteria = {}\n        if not isinstance(criteria, Mapping):\n            raise ValueError(\"noul criteria must be a mapping when provided\")\n        criteria = {str(k).lower(): v for k, v in criteria.items()}\n\n    instructions = question.get(\"instructions\", question.get(\"ins\", \"\"))\n    if not isinstance(instructions, str):\n        instructions = json.dumps(instructions, ensure_ascii=False, default=str)\n\n    result = {\"t\": qtype, \"ins\": instructions, \"crit\": criteria}\n    if \"labels\" in question:\n        result[\"labels\"] = dict(question[\"labels\"])\n    return result\n\n\ndef render_options(question: Mapping[str, Any]) -> list[str]:\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    qtype, criteria = q[\"t\"], q.get(\"crit\")\n    if qtype == \"choice\":\n        return [\n            str(k) if v is None or v == \"\" else f\"{k}: {_render_value(v)}\"\n            for k, v in criteria.items()\n        ]\n    if qtype == \"score\":\n        return [f\"level {i}: {_render_value(v)}\" for i, v in enumerate(criteria)]\n\n    labels = q.get(\"labels\", _DEFAULT_NOUL_LABELS)\n    false_label = str(labels.get(\"false\", \"false\")).strip() or \"false\"\n    true_label = str(labels.get(\"true\", \"true\")).strip() or \"true\"\n    false_desc = criteria.get(\"false\")\n    true_desc = criteria.get(\"true\")\n    return [\n        f\"{false_label}: \" + (\n            _render_value(false_desc) if false_desc not in (None, \"\") else \"no, the statement does not hold\"\n        ),\n        f\"{true_label}: \" + (\n            _render_value(true_desc) if true_desc not in (None, \"\") else \"yes, the statement holds\"\n        ),\n    ]\n\n\ndef build_sequence(\n    tokenizer,\n    state: str | dict | list,\n    question: Mapping[str, Any],\n    *,\n    max_len: int = 512,\n    head_max_len: int = 192,\n    truncate_left: bool = False,\n    state_ids: Sequence[int] | None = None,\n) -> tuple[list[int], list[int]]:\n    \"\"\"Build a compact typed-decision sequence and option marker positions.\"\"\"\n    q = normalize_question(question) if \"t\" not in question else dict(question)\n    options = render_options(q)\n    mask_token = tokenizer.mask_token\n    instruction = str(q[\"ins\"]).replace(mask_token, \" \")\n    head_ids = tokenizer(\n        f\"{q['t']} question: {instruction}\", add_special_tokens=False\n    )[\"input_ids\"]\n\n    option_ids: list[list[int]] = []\n    for option in options:\n        ids = tokenizer(\n            \" \" + option.replace(mask_token, \" \"),\n            add_special_tokens=False,\n            truncation=True,\n            max_length=48,\n        )[\"input_ids\"]\n        option_ids.append([tokenizer.mask_token_id] + ids)\n\n    option_budget = head_max_len - sum(len(x) for x in option_ids)\n    if option_budget < 16:\n        per = max(4, (head_max_len - 16) // max(1, len(option_ids)))\n        option_ids = [x[:per] for x in option_ids]\n        option_budget = head_max_len - sum(len(x) for x in option_ids)\n    head_ids = head_ids[: max(8, option_budget)]\n\n    ids = [tokenizer.cls_token_id] + head_ids + [tokenizer.sep_token_id]\n    markers: list[int] = []\n    for option in option_ids:\n        markers.append(len(ids))\n        ids.extend(option)\n    ids.append(tokenizer.sep_token_id)\n\n    if state_ids is None:\n        text = _serialize_state(state).replace(mask_token, \" \")\n        state_ids = tokenizer(text, add_special_tokens=False)[\"input_ids\"]\n    room = max(0, max_len - len(ids) - 1)\n    if truncate_left:\n        start = max(0, len(state_ids) - room)\n        state_part = list(state_ids[start:])\n    else:\n        state_part = list(state_ids[:room])\n    ids = (ids + state_part + [tokenizer.sep_token_id])[:max_len]\n    markers = [m for m in markers if m < max_len]\n    return ids, markers\n\n\ndef collate_items(items: Sequence[Mapping[str, Any]], pad_id: int) -> dict[str, Tensor]:\n    if not items:\n        raise ValueError(\"items must not be empty\")\n    n = len(items)\n    length = max(len(x[\"ids\"]) for x in items)\n    kmax = max(len(x[\"markers\"]) for x in items)\n    ids = torch.full((n, length), int(pad_id), dtype=torch.long)\n    mask = torch.zeros((n, length), dtype=torch.long)\n    marker_pos = torch.zeros((n, kmax), dtype=torch.long)\n    marker_mask = torch.zeros((n, kmax), dtype=torch.bool)\n    qtype = torch.zeros(n, dtype=torch.long)\n\n    for i, item in enumerate(items):\n        row = torch.tensor(item[\"ids\"], dtype=torch.long)\n        ids[i, : row.numel()] = row\n        mask[i, : row.numel()] = 1\n        m = torch.tensor(item[\"markers\"], dtype=torch.long)\n        marker_pos[i, : m.numel()] = m\n        marker_mask[i, : m.numel()] = True\n        qtype[i] = int(item[\"qtype\"])\n    return {\n        \"input_ids\": ids,\n        \"attention_mask\": mask,\n        \"marker_pos\": marker_pos,\n        \"marker_mask\": marker_mask,\n        \"qtype\": qtype,\n    }\n\n\n\ndef _apply_rope_compat(cfg):\n    rope = getattr(cfg, \"rope_parameters\", None)\n    if not isinstance(rope,dict):\n        return\n    flat = rope.get(\"rope_theta\")\n    for layer_type, attr in ((\"full_attention\",\"global_rope_theta\"),\n                             (\"sliding_attention\",\"local_rope_theta\")):\n        v = rope.get(layer_type)\n        theta = v.get(\"rope_theta\") if isinstance(v,dict) else flat\n        if theta is not None and hasattr(cfg,attr):\n            setattr(cfg,attr,float(theta))\n\n\nclass LayaStandalone(nn.Module):\n    \"\"\"Native pretrained ModernBERT + exact 64-token option-scoring head.\n\n    ALL weights are stored locally in model.safetensors. No auxiliary act_head.\n    \"\"\"\n    def __init__(self, encoder_config=None, *, budget=64, head_layers=2,\n                 pretrained_components=None):\n        super().__init__()\n        self.budget = int(budget)\n        assert self.budget == 64, \"This checkpoint implements the validated 64-token head\"\n        if pretrained_components is not None:\n            # EXPORT ONLY: reuse the loaded upstream model without duplicating\n            # ~400M weights; this branch is not needed during standalone load.\n            self.encoder = pretrained_components.encoder\n            self.head = pretrained_components.head\n            self.type_emb = pretrained_components.type_emb\n            self.scorer = pretrained_components.scorer\n        else:\n            assert encoder_config is not None\n            _apply_rope_compat(encoder_config)\n            self.encoder = AutoModel.from_config(\n                encoder_config, attn_implementation=\"sdpa\")\n            d = int(encoder_config.hidden_size)\n            nhead = max(1,d//64)\n            layer = nn.TransformerEncoderLayer(\n                d,nhead,dim_feedforward=4*d,dropout=0.1,\n                batch_first=True,norm_first=True)\n            self.head = nn.TransformerEncoder(\n                layer,int(head_layers),enable_nested_tensor=False)\n            self.type_emb = nn.Embedding(3,d)\n            self.scorer = nn.Sequential(nn.LayerNorm(d),nn.Linear(d,d),\n                                        nn.GELU(),nn.Linear(d,1))\n\n    def _select_head_tokens(self,h,attention_mask,marker_pos,marker_mask):\n        # Copied from V4.6, keeping exact order/masks for reproduction.\n        b,t,d=h.shape\n        cls=torch.zeros((b,1),dtype=torch.long,device=h.device)\n        lm_count=min(self.budget,max(t-1,1))\n        lm=torch.linspace(1,max(t-1,1),steps=lm_count,device=h.device).round().long()\n        lm=lm.clamp(max=t-1).view(1,-1).expand(b,-1)\n        marker_ok=marker_mask.bool()\n        lm_ok=attention_mask.bool().gather(1,lm)\n        lm_ok=lm_ok & (lm!=0)\n        duplicate=(lm[:,:,None]==marker_pos[:,None,:]) & marker_ok[:,None,:]\n        lm_ok=lm_ok & ~duplicate.any(-1)\n        idx=torch.cat((cls,marker_pos.clamp(0,t-1),lm),dim=1)\n        keep=torch.cat((torch.ones((b,1),dtype=torch.bool,device=h.device),\n                        marker_ok,lm_ok),dim=1)\n        gathered=h.gather(1,idx[:,:,None].expand(-1,-1,d))\n        return gathered,keep\n\n    def forward(self,input_ids,attention_mask,marker_pos,marker_mask,qtype):\n        h=self.encoder(input_ids=input_ids,\n                       attention_mask=attention_mask).last_hidden_state\n        h=h+self.type_emb(qtype)[:,None,:]\n        compact,keep=self._select_head_tokens(\n            h,attention_mask,marker_pos,marker_mask)\n        transformed=self.head(compact,src_key_padding_mask=~keep)\n        marker_hidden=transformed[:,1:1+marker_pos.shape[1],:]\n        logits=self.scorer(marker_hidden).squeeze(-1).float()\n        logits=logits.masked_fill(~marker_mask,-1e4)\n        return logits,None\n\n    @torch.inference_mode()\n    def predict(self,tokenizer,state,questions,*,max_len=512,head_max_len=192):\n        items=[];names=[];labels=[]\n        for name,raw in questions.items():\n            q=normalize_question(raw)\n            if q[\"t\"]!=\"choice\":\n                raise ValueError(\"Only choice questions supported by predict()\")\n            opts=list(q[\"crit\"].keys())\n            ids,markers=build_sequence(tokenizer,state,q,max_len=max_len,\n                                       head_max_len=head_max_len)\n            if len(markers)!=len(opts):\n                raise ValueError(\"Truncated an option marker\")\n            names.append(name);labels.append(opts)\n            items.append(dict(ids=ids,markers=markers,qtype=QTYPES[q[\"t\"]]))\n        batch=collate_items(items,int(tokenizer.pad_token_id))\n        device=next(self.parameters()).device\n        batch={k:v.to(device) for k,v in batch.items()}\n        with (torch.autocast(\"cuda\",dtype=torch.float16)\n              if device.type==\"cuda\" else __import__(\"contextlib\").nullcontext()):\n            logits,_=self(**batch)\n        probs=logits.float().masked_fill(~batch[\"marker_mask\"],-1e9).softmax(-1).cpu()\n        answers={}\n        for i,name in enumerate(names):\n            p=probs[i,:len(labels[i])]\n            best=int(p.argmax())\n            answers[name]=dict(choice=labels[i][best],\n                               confidence=float(p[best]),\n                               probabilities={label:float(pr) for label,pr in zip(labels[i],p)})\n        return answers\n\n\ndef load_model(model_dir,*,device=\"cpu\",local_files_only=True,compile_head=False):\n    \"\"\"Load entirely from the local standalone directory. No network needed.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text())\n    assert metadata[\"format\"]==\"laya-1p2-full-standalone-v1\"\n    enc_cfg=AutoConfig.from_pretrained(\n        str(root/\"encoder\"),local_files_only=local_files_only)\n    model=LayaStandalone(enc_cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(\n        str(root/\"tokenizer\"),local_files_only=local_files_only)\n    return model,tok\n\ndef local_pair_indices(model):\n    \"\"\"Verify FULL / same-window LOCAL / LOCAL triples in ModernBERT.\n\n    transformers==4.57.6 uses attn.local_attention: (-1,-1) means FULL,\n    (64,64) is a 128-token sliding window. Newer builds can expose\n    config.layer_types with attn.sliding_window instead. Support both.\n    \"\"\"\n    layers=model.encoder.layers\n    cfg=model.encoder.config\n    n=len(layers)\n    layer_types=getattr(cfg,\"layer_types\",None)\n    if layer_types is not None and len(layer_types)!=n:\n        raise ValueError(\"ModernBERT layer_types count does not match layers\")\n\n    def mode_at(i):\n        a=layers[i].attn\n        old_window=getattr(a,\"local_attention\",None)\n        if old_window is not None:\n            window=tuple(old_window)\n            if window==(-1,-1):\n                return (\"full\",None)\n            if len(window)!=2 or min(window)<0:\n                raise ValueError(f\"Malformed local attention window at {i}: {window}\")\n            return (\"local\",window)\n        typ=(layer_types[i] if layer_types is not None else\n             getattr(layers[i],\"attention_type\",None))\n        window=getattr(a,\"sliding_window\",None)\n        if typ==\"full_attention\" and window is None:\n            return (\"full\",None)\n        if typ==\"sliding_attention\" and window is not None:\n            return (\"local\",int(window))\n        raise ValueError(f\"Unsupported ModernBERT attention metadata at layer {i}: {typ}, {window}\")\n\n    pairs=[]\n    for anchor in range(0,n,3):\n        mode,_=mode_at(anchor)\n        if mode!=\"full\":\n            raise ValueError(f\"Layer {anchor} must be FULL attention\")\n        left_idx,right_idx=anchor+1,anchor+2\n        if right_idx>=n:\n            continue\n        m1,w1=mode_at(left_idx)\n        m2,w2=mode_at(right_idx)\n        if m1!=\"local\" or m2!=\"local\" or w1!=w2:\n            raise ValueError(\n                f\"Layers {(left_idx,right_idx)} must have identical local windows: {m1}/{w1}, {m2}/{w2}\")\n        pairs.append((left_idx,right_idx))\n    return pairs\n\n\nclass SVDDeltaLinear(nn.Module):\n    \"\"\"Second-layer linear projection = shared first-layer projection + A @ B.\n\n    The shared linear is a plain Python reference, NOT a registered submodule;\n    its weights appear only once in state_dict, under the original first layer.\n    \"\"\"\n    def __init__(self, first_linear, rank, *, A=None, B=None):\n        super().__init__()\n        if not isinstance(first_linear,nn.Linear):\n            raise TypeError(f\"Expected nn.Linear, got {type(first_linear)}\")\n        out_dim,in_dim=first_linear.weight.shape\n        rank=int(rank)\n        if not (1<=rank<min(in_dim,out_dim)):\n            raise ValueError(f\"Unsupported rank {rank} for {out_dim}x{in_dim}\")\n        # Using object.__setattr__ prevents double registering the shared module.\n        object.__setattr__(self,\"_shared\",first_linear)\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.max_rank=rank\n        self.active_rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),device=first_linear.weight.device,\n                                        dtype=first_linear.weight.dtype))\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A.to(self.A.device,self.A.dtype))\n                self.B.copy_(B.to(self.B.device,self.B.dtype))\n\n    def set_rank(self, rank):\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(f\"rank {rank} outside 1..{self.max_rank}\")\n        self.active_rank=rank\n\n    def shrink(self, rank):\n        \"\"\"Permanently discard extra SVD columns for a compact checkpoint.\"\"\"\n        rank=int(rank)\n        if rank<1 or rank>self.max_rank:\n            raise ValueError(rank)\n        self.A=nn.Parameter(self.A[:,:rank].detach().clone())\n        self.B=nn.Parameter(self.B[:rank,:].detach().clone())\n        self.max_rank=rank\n        self.active_rank=rank\n\n    def forward(self,x):\n        # fp16 autocast handles both paths identically, with FP32 stored weights.\n        w=self._shared.weight\n        bias=self._shared.bias\n        base=F.linear(x,w,bias)\n        r=self.active_rank\n        return base+F.linear(F.linear(x,self.B[:r,:]),self.A[:,:r])\n\n\n\n# V4.9 does NOT force compression of all nine local pairs.\n# Each selected follower projection gets a separate low-rank residual;\n# all unselected attention projection matrices retain pretrained weights.\ndef projection_slots(model):\n    slots=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            a=getattr(model.encoder.layers[first].attn,component)\n            b=getattr(model.encoder.layers[second].attn,component)\n            if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n                raise TypeError(f\"Expected nn.Linear for layer {second}.{component}\")\n            if a.weight.shape != b.weight.shape:\n                raise ValueError(f\"Incompatible original projection: {second}.{component}\")\n            if a.bias is not None or b.bias is not None:\n                raise ValueError(f\"Biasful projection needs separate residual bias: {second}.{component}\")\n            slots.append((first,second,component))\n    return slots\n\n\ndef _validate_slot(model, first, second, component):\n    \"\"\"Validate schedule independently of other already-compressed slots.\"\"\"\n    first,second=int(first),int(second)\n    component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Unknown local pair: {(first,second)}\")\n    if component not in (\"Wqkv\",\"Wo\"):\n        raise ValueError(f\"Unknown projection: {component}\")\n    return first,second,component\n\n\ndef svd_factors(model, first, second, component, max_rank=256,\n                oversampling=8, niter=2):\n    \"\"\"Compute one factorization per projection from ORIGINAL pretrained weights.\n\n    A and B are float32 CPU tensors and can be sliced for different ranks.\n    \"\"\"\n    first,second,component=_validate_slot(model,first,second,component)\n    a=getattr(model.encoder.layers[first].attn,component)\n    b=getattr(model.encoder.layers[second].attn,component)\n    if not isinstance(a,nn.Linear) or not isinstance(b,nn.Linear):\n        raise ValueError(\"SVD must be done BEFORE that follower is compressed\")\n    with torch.no_grad():\n        delta=b.weight.detach().float()-a.weight.detach().float()\n        r=min(int(max_rank),min(delta.shape)-1)\n        q=min(r+int(oversampling),min(delta.shape))\n        # Randomized SVD (approximation, not an exact mathematical SVD).\n        U,S,V=torch.svd_lowrank(delta,q=q,niter=int(niter))\n        root=S[:r].clamp_min(0).sqrt()\n        A=(U[:,:r]*root.unsqueeze(0)).cpu().contiguous()\n        B=(root.unsqueeze(1)*V[:,:r].T).cpu().contiguous()\n        total=float(delta.square().sum().item())\n        error=float((delta-(A.to(delta.device)@B.to(delta.device))).square().sum().item())\n        del U,S,V,delta\n    return A,B,{\"rank\":r,\n        \"relative_frobenius_error\":(error/max(total,1e-30))**0.5,\n        \"saved_parameter_bytes_if_rank256\":None}\n\n\ndef build_adapter(model, first, second, component, rank, factors=None,\n                  *, initialized=True):\n    \"\"\"Replace ONE follower's projection; return original module for rollback.\n\n    If initialized=False, A/B are zeros until strictly loaded from checkpoint.\n    \"\"\"\n    first=int(first);second=int(second);component=str(component)\n    if (first,second) not in local_pair_indices(model):\n        raise ValueError(f\"Invalid local pair: {(first,second)}\")\n    leader=getattr(model.encoder.layers[first].attn,component)\n    follower_block=model.encoder.layers[second].attn\n    old=getattr(follower_block,component)\n    if not isinstance(leader,nn.Linear) or not isinstance(old,nn.Linear):\n        raise TypeError(\"Projection already wrapped or unavailable\")\n    if leader.bias is not None or old.bias is not None:\n        raise ValueError(\"Biasful attention projections are unsupported\")\n    if initialized:\n        if factors is None:\n            A,B,_=svd_factors(model,first,second,component,max_rank=rank)\n        else:\n            A,B=factors\n        adapter=SVDDeltaLinear(leader,rank,A=A[:,:rank],B=B[:rank,:])\n    else:\n        adapter=SVDDeltaLinear(leader,rank)\n    setattr(follower_block,component,adapter)\n    return old\n\n\ndef restore_original_projection(model, second, component, original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Rollback needs the original pretrained linear module\")\n    setattr(model.encoder.layers[int(second)].attn,str(component),original)\n\n\ndef selected_adapters(model):\n    selected=[]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            layer=getattr(model.encoder.layers[second].attn,component)\n            if isinstance(layer,SVDDeltaLinear):\n                selected.append((first,second,component,layer))\n    return selected\n\n\ndef unique_parameter_count(model):\n    return sum(p.numel() for p in model.parameters())\n\n\ndef estimated_saved_bytes(model, first, second, component, rank):\n    leader=getattr(model.encoder.layers[first].attn,component)\n    outdim,indim=leader.weight.shape\n    # Full follower weight removed; A (out×rank), B (rank×in) remain.\n    return 4*(outdim*indim-int(rank)*(outdim+indim))\n\n\ndef load_v49_model(model_dir, *, device=\"cpu\", compile_head=False):\n    \"\"\"Strict, fully offline restoration of selectively compressed model.\"\"\"\n    root=Path(model_dir).resolve()\n    meta=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if meta[\"format\"]!=\"laya-v49-selective-svd-v1\":\n        raise ValueError(\"Wrong model format: expected V4.9 selective SVD\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(meta[\"head_context_budget\"]),\n                         head_layers=int(meta[\"head_layers\"]))\n    for item in meta[\"compressed_projections\"]:\n        build_adapter(model,int(item[\"first\"]),int(item[\"second\"]),\n                      str(item[\"component\"]),int(item[\"rank\"]),\n                      initialized=False)\n    weights=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(weights,strict=True)\n    # No duplicated original follower weights or duplicated leader submodule.\n    assert len(selected_adapters(model))==len(meta[\"compressed_projections\"])\n    for first,second,component,wrapper in selected_adapters(model):\n        leader=getattr(model.encoder.layers[first].attn,component)\n        assert wrapper._shared is leader\n        assert \"_shared\" not in wrapper._modules\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n\n\n# ---------------------------------------------------------------------------\n# V5.0: brute-force all 28 ModernBERT blocks, attention + FFN, preserving the\n# proven V4.9 pretrained-pair DELTA path as one selectable candidate family.\n# ---------------------------------------------------------------------------\n\nclass IndependentSVDLinear(nn.Module):\n    \"\"\"Plain original projection compressed as A @ B; original dense W omitted.\n\n    Compatible with ModernBERT's attention Wqkv, Wo and MLP Wi, Wo calls.\n    A saved bias is kept if the original linear was biased.\n    \"\"\"\n    def __init__(self, original: nn.Linear, rank: int, A=None, B=None):\n        super().__init__()\n        if not isinstance(original, nn.Linear):\n            raise TypeError(\"Expected original nn.Linear\")\n        out_dim,in_dim=original.weight.shape\n        rank=int(rank)\n        if rank<=0 or rank>=min(out_dim,in_dim):\n            raise ValueError(f\"Bad SVD rank {rank} for {out_dim}x{in_dim}\")\n        self.in_features=int(in_dim)\n        self.out_features=int(out_dim)\n        self.rank=rank\n        self.A=nn.Parameter(torch.zeros((out_dim,rank),\n            dtype=original.weight.dtype,device=original.weight.device))\n        self.B=nn.Parameter(torch.zeros((rank,in_dim),\n            dtype=original.weight.dtype,device=original.weight.device))\n        if original.bias is None:\n            self.register_parameter(\"bias\",None)\n        else:\n            self.bias=nn.Parameter(original.bias.detach().clone())\n        if A is not None and B is not None:\n            with torch.no_grad():\n                self.A.copy_(A[:,:rank].to(device=self.A.device,dtype=self.A.dtype))\n                self.B.copy_(B[:rank,:].to(device=self.B.device,dtype=self.B.dtype))\n\n    def forward(self,x):\n        return F.linear(F.linear(x,self.B),self.A,self.bias)\n\n\ndef all_encoder_slots(model):\n    \"\"\"Enumerate available projections in EVERY pretrained encoder layer.\"\"\"\n    slots=[]\n    for index,layer in enumerate(model.encoder.layers):\n        for block,names in ((\"attn\",(\"Wqkv\",\"Wo\")),(\"mlp\",(\"Wi\",\"Wo\"))):\n            component=getattr(layer,block)\n            for name in names:\n                module=getattr(component,name,None)\n                if module is None:\n                    raise ValueError(f\"Required projection layer {index}.{block}.{name} missing\")\n                if not isinstance(module,nn.Linear):\n                    raise TypeError(f\"Not an uncompressed nn.Linear: {index}.{block}.{name}: {type(module)}\")\n                slots.append((index,block,name))\n    if len(slots)!=4*len(model.encoder.layers):\n        raise RuntimeError(\"Expected exactly two attention and two FFN projections per layer\")\n    return slots\n\n\ndef list_candidate_families(model):\n    \"\"\"Full brute-force search space: 112 own-SVD + 18 delta-shared slots.\"\"\"\n    options=[{\"layer\":int(layer),\"block\":block,\"component\":name,\n              \"mode\":\"own_svd\"}\n             for layer,block,name in all_encoder_slots(model)]\n    for first,second in local_pair_indices(model):\n        for component in (\"Wqkv\",\"Wo\"):\n            options.append({\"layer\":int(second),\"leader\":int(first),\n                            \"block\":\"attn\",\"component\":component,\n                            \"mode\":\"pair_delta\"})\n    return options\n\n\ndef own_svd_factors(original, *, max_rank=256, oversampling=8,niter=2):\n    \"\"\"Randomized, truncated SVD of ORIGINAL pretrained W (not layer delta).\"\"\"\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Expected an uncompressed linear projection\")\n    with torch.no_grad():\n        weight=original.weight.detach().float()\n        rank=min(int(max_rank),min(weight.shape)-1)\n        q=min(rank+int(oversampling),min(weight.shape))\n        U,S,V=torch.svd_lowrank(weight,q=q,niter=int(niter))\n        scale=S[:rank].clamp_min(0).sqrt()\n        A=(U[:,:rank]*scale[None,:]).cpu().contiguous()\n        B=(scale[:,None]*V[:,:rank].T).cpu().contiguous()\n        approximated=A.to(weight.device)@B.to(weight.device)\n        numerator=float((weight-approximated).square().sum().item())\n        denominator=float(weight.square().sum().item())\n    return A,B,{\"rank\":rank,\n                \"relative_frobenius_error\":(numerator/max(denominator,1e-30))**.5}\n\n\ndef projection_at(model, spec):\n    return getattr(getattr(model.encoder.layers[int(spec[\"layer\"])],\n                           str(spec[\"block\"])),str(spec[\"component\"]))\n\n\ndef replace_projection(model, spec, *, factors=None, initialized=True):\n    \"\"\"Apply one projection replacement; return original module for rollback.\"\"\"\n    idx=int(spec[\"layer\"])\n    block=str(spec[\"block\"])\n    comp=str(spec[\"component\"])\n    rank=int(spec[\"rank\"])\n    mode=str(spec[\"mode\"])\n    owner=getattr(model.encoder.layers[idx],block)\n    old=getattr(owner,comp)\n    if not isinstance(old,nn.Linear):\n        raise TypeError(f\"Already compressed or unsupported {idx}.{block}.{comp}\")\n    if mode==\"own_svd\":\n        if initialized and factors is None:\n            factors=own_svd_factors(old,max_rank=rank)[:2]\n        if initialized:\n            A,B=factors\n            replacement=IndependentSVDLinear(old,rank,A,B)\n        else:\n            replacement=IndependentSVDLinear(old,rank)\n    elif mode==\"pair_delta\":\n        if block!=\"attn\" or comp not in (\"Wqkv\",\"Wo\"):\n            raise ValueError(\"Pair delta only supported for local attention\")\n        leader=int(spec[\"leader\"])\n        if (leader,idx) not in local_pair_indices(model):\n            raise ValueError(f\"Not a valid local pair: {leader},{idx}\")\n        shared=getattr(model.encoder.layers[leader].attn,comp)\n        if not isinstance(shared,nn.Linear):\n            raise TypeError(\"Leader must retain original pretrained linear\")\n        if shared.bias is not None or old.bias is not None:\n            raise ValueError(\"Pair delta only supported for bias-free attention\")\n        if initialized:\n            if factors is None:\n                A,B,_=svd_factors(model,leader,idx,comp,max_rank=rank)\n            else:\n                A,B=factors\n            replacement=SVDDeltaLinear(shared,rank,A=A[:,:rank],B=B[:rank,:])\n        else:\n            replacement=SVDDeltaLinear(shared,rank)\n    else:\n        raise ValueError(f\"Unsupported family: {mode}\")\n    setattr(owner,comp,replacement)\n    return old\n\n\ndef revert_projection(model,spec,original):\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Original projection must be nn.Linear\")\n    owner=getattr(model.encoder.layers[int(spec[\"layer\"])],str(spec[\"block\"]))\n    setattr(owner,str(spec[\"component\"]),original)\n\n\ndef applied_adapters(model,selected_specs):\n    result=[]\n    for spec in selected_specs:\n        wrapper=projection_at(model,spec)\n        if not isinstance(wrapper,(IndependentSVDLinear,SVDDeltaLinear)):\n            raise AssertionError(f\"Missing compressed adapter for {spec}\")\n        result.append(wrapper)\n    return result\n\n\ndef expected_saved_bytes(model,spec):\n    \"\"\"FP32 parameter saving for one replacement, including any old bias.\"\"\"\n    idx=int(spec[\"layer\"])\n    owner=getattr(model.encoder.layers[idx],str(spec[\"block\"]))\n    original=getattr(owner,str(spec[\"component\"]))\n    if not isinstance(original,nn.Linear):\n        raise TypeError(\"Call before installing candidate\")\n    out_dim,in_dim=original.weight.shape\n    rank=int(spec[\"rank\"])\n    # Bias exists in both old and new for own_svd; cancels out.\n    # Pair delta has no bias.\n    return int(4*(out_dim*in_dim-rank*(out_dim+in_dim)))\n\n\ndef load_v50_model(model_dir, *, device=\"cpu\",compile_head=False):\n    \"\"\"Strict reload of all hybrid low-rank projections from local disk only.\"\"\"\n    root=Path(model_dir).resolve()\n    info=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if info[\"format\"]!=\"laya-v50-all28-bruteforce-v1\":\n        raise ValueError(\"Not a V5.0 hybrid model\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(info[\"head_context_budget\"]),\n                         head_layers=int(info[\"head_layers\"]))\n    # Rebuild in the same order: only uncompressed leaders are valid for delta.\n    for spec in info[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    full=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(full,strict=True)\n    for spec,adapter in zip(info[\"compressed_projections\"],\n                            applied_adapters(model,info[\"compressed_projections\"])):\n        if spec[\"mode\"]==\"pair_delta\":\n            leader=getattr(model.encoder.layers[int(spec[\"leader\"])].attn,\n                           str(spec[\"component\"]))\n            assert adapter._shared is leader\n            assert \"_shared\" not in adapter._modules\n    model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tok=AutoTokenizer.from_pretrained(str(root/\"tokenizer\"),local_files_only=True)\n    return model,tok\n\n\n# ================================================================\n# Laya V5.3: CeNN-inspired token-neighborhood liquid recurrence\n# ================================================================\n\nclass LiquidCeNNProjection(nn.Module):\n    \"\"\"Replace/augment an FFN output projection with a 1D liquid CeNN cell.\n\n    This is CeNN-*inspired*: local nearest-neighbor coupling via depthwise\n    Conv1d over tokens plus gated state relaxation. It is not the original\n    continuous-time Chua CeNN equation, nor an LTC ODE solver.\n\n    Modes:\n      * replace: remove the old projection entirely, f(x)=liquid(x).\n      * residual: keep the original projection, f(x)=base(x)+scale*liquid(x).\n\n    The residual scale is initially zero, so this mode is exactly the\n    pretrained baseline at creation time, including identical predictions.\n    \"\"\"\n    def __init__(self,original,width=64,steps=3,mode=\"residual\",\n                 dt_bound=0.50):\n        super().__init__()\n        if not hasattr(original,\"in_features\") or not hasattr(original,\"out_features\"):\n            raise TypeError(\"LiquidCeNN requires a linear-like FFN projection\")\n        self.in_features=int(original.in_features)\n        self.out_features=int(original.out_features)\n        self.width=int(width)\n        self.steps=int(steps)\n        self.mode=str(mode)\n        self.dt_bound=float(dt_bound)\n        if self.width<4 or self.steps<1 or self.steps>10:\n            raise ValueError(\"Bad cell width or recurrent step count\")\n        if self.mode not in (\"replace\",\"residual\"):\n            raise ValueError(self.mode)\n        if not (0.0<self.dt_bound<=1.0):\n            raise ValueError(\"dt_bound must be in (0,1]\")\n        # Only register the original dense/low-rank projection in residual\n        # mode; replacement genuinely deletes its weights from state_dict.\n        self.base=original if self.mode==\"residual\" else None\n        self.drive=nn.Linear(self.in_features,self.width,bias=True)\n        self.drive_norm=nn.LayerNorm(self.width)\n        self.neighbor=nn.Conv1d(self.width,self.width,kernel_size=3,\n                                padding=1,groups=self.width,bias=False)\n        self.recurrent=nn.Linear(self.width,self.width,bias=False)\n        self.liquid_gate=nn.Linear(2*self.width,self.width)\n        self.readout=nn.Linear(self.width,self.out_features,bias=True)\n        # Start with small dynamic corrections instead of random large logits.\n        nn.init.normal_(self.readout.weight,mean=0.,std=0.008)\n        nn.init.zeros_(self.readout.bias)\n        self.scale=nn.Parameter(torch.tensor(0.,dtype=torch.float32))\n\n        # Match original device and dtype without moving the original base.\n        original_parameter=next(original.parameters(),None)\n        if original_parameter is not None:\n            for module in [self.drive,self.drive_norm,self.neighbor,\n                           self.recurrent,self.liquid_gate,self.readout]:\n                module.to(device=original_parameter.device,dtype=original_parameter.dtype)\n\n    def liquid(self,x):\n        # x must have [batch, tokens, d_ffn]. Spatial nearest neighbors\n        # are token neighbors, not a two-dimensional image lattice.\n        if x.ndim!=3:\n            raise ValueError(f\"Expected [B,T,D] FFN activation, got {tuple(x.shape)}\")\n        u=torch.tanh(self.drive_norm(self.drive(x)))\n        h=torch.zeros_like(u)\n        for _ in range(self.steps):\n            neighbors=self.neighbor(h.transpose(1,2)).transpose(1,2)\n            update_gate=torch.sigmoid(self.liquid_gate(torch.cat((u,h),dim=-1)))\n            dt=self.dt_bound*update_gate\n            proposal=torch.tanh(u+neighbors+self.recurrent(h))\n            h=h+dt*(proposal-h)\n        return self.readout(h)\n\n    def forward(self,x):\n        correction=self.liquid(x)\n        if self.mode==\"replace\":\n            return correction\n        # Alpha=0 is an EXACT base-model control despite random CeNN weights.\n        return self.base(x)+self.scale.to(dtype=correction.dtype)*correction\n\n\ndef install_liquid_ffn(model,layer_index,*,width=64,steps=3,mode=\"residual\",\n                       dt_bound=0.5):\n    layer_index=int(layer_index)\n    if not (0<=layer_index<len(model.encoder.layers)):\n        raise IndexError(layer_index)\n    owner=model.encoder.layers[layer_index].mlp\n    previous=owner.Wo\n    if isinstance(previous,LiquidCeNNProjection):\n        raise ValueError(f\"Liquid layer {layer_index} already installed\")\n    candidate=LiquidCeNNProjection(previous,width,steps,mode,dt_bound)\n    owner.Wo=candidate\n    return previous\n\n\ndef uninstall_liquid_ffn(model,layer_index,previous):\n    layer_index=int(layer_index)\n    if not isinstance(previous,nn.Module):\n        raise TypeError(\"Replacement rollback must restore nn.Module\")\n    module=model.encoder.layers[layer_index].mlp\n    if not isinstance(module.Wo,LiquidCeNNProjection):\n        raise ValueError(\"Current layer is not LiquidCeNNProjection\")\n    module.Wo=previous\n\n\ndef find_liquid_cells(model):\n    return [(i,block.mlp.Wo)\n            for i,block in enumerate(model.encoder.layers)\n            if isinstance(block.mlp.Wo,LiquidCeNNProjection)]\n\n\ndef load_v53_model(model_dir, *,device=\"cpu\",compile_head=False):\n    \"\"\"Fully offline strict reconstruction of compressed V5.3 + liquid CeNN.\"\"\"\n    root=Path(model_dir).resolve()\n    metadata=json.loads((root/\"config.json\").read_text(encoding=\"utf-8\"))\n    if metadata.get(\"format\")!=\"laya-v53-liquid-cenn-v1\":\n        raise ValueError(\"Not a V5.3 liquid CeNN export\")\n    cfg=AutoConfig.from_pretrained(str(root/\"encoder\"),local_files_only=True)\n    model=LayaStandalone(cfg,budget=int(metadata[\"head_context_budget\"]),\n                         head_layers=int(metadata[\"head_layers\"]))\n    for spec in metadata[\"compressed_projections\"]:\n        replace_projection(model,spec,initialized=False)\n    spec=metadata[\"liquid_cell\"]\n    install_liquid_ffn(model,int(spec[\"layer\"]),width=int(spec[\"width\"]),\n       steps=int(spec[\"steps\"]),mode=str(spec[\"mode\"]),\n       dt_bound=float(spec[\"dt_bound\"]))\n    state=load_file(str(root/\"model.safetensors\"),device=\"cpu\")\n    model.load_state_dict(state,strict=True)\n    assert len(find_liquid_cells(model))==1\n    model=model.to(device).eval().requires_grad_(False)\n    model.encoder.config.reference_compile=False\n    if compile_head:\n        model.head=torch.compile(model.head,mode=\"reduce-overhead\",dynamic=False)\n    tokenizer=AutoTokenizer.from_pretrained(\n       str(root/\"tokenizer\"),local_files_only=True)\n    return model,tokenizer\n"
runtime_file=OUT/"modeling_laya_v53.py"
runtime_file.write_text(V53_RUNTIME_CODE,encoding="utf-8")
spec=importlib.util.spec_from_file_location("v53_runtime",str(runtime_file))
assert spec and spec.loader
v53=importlib.util.module_from_spec(spec)
spec.loader.exec_module(v53)
normalize_question=v53.normalize_question
build_sequence=v53.build_sequence
collate_items=v53.collate_items
QTYPES=v53.QTYPES
teacher,_=v53.load_model(parent_dir,device="cuda")
student,_=v53.load_model(parent_dir,device="cuda")
teacher.eval().requires_grad_(False)
student.eval().requires_grad_(False)
teacher.encoder.config.reference_compile=False
student.encoder.config.reference_compile=False
native_unique=v53.unique_parameter_count(teacher)
assert len(student.encoder.layers)==28
assert len(v53.all_encoder_slots(student))==112
# Short liquid cell unit test: residual starts IDENTICAL and true replacement
# physically omits the old base module. Tests B,T,D shapes and gradients.
with torch.enable_grad():
    simple=nn.Linear(80,16,bias=False).to(DEVICE)
    sample=torch.randn(2,11,80,device=DEVICE)
    residual_cell=v53.LiquidCeNNProjection(
        simple,width=8,steps=2,mode="residual").to(DEVICE)
    torch.testing.assert_close(residual_cell(sample),simple(sample),atol=1e-6,rtol=1e-6)
    replacement_cell=v53.LiquidCeNNProjection(
        simple,width=8,steps=2,mode="replace").to(DEVICE)
    assert replacement_cell.base is None
    assert not any(name.startswith("base.") for name in replacement_cell.state_dict())
    result=replacement_cell(sample)
    assert result.shape==(2,11,16) and bool(torch.isfinite(result).all())
    result.square().mean().backward()
    assert replacement_cell.drive.weight.grad is not None
    del simple,sample,residual_cell,replacement_cell,result
gc.collect()
print("PASS: Liquid-CeNN residual identity, replacement weight removal and gradients")


PASS: Liquid-CeNN residual identity, replacement weight removal and gradients


In [4]:
#@title 4. CASE-DISJOINT TRAIN / PROBE / AUDIT benchmark from V4.6
def _jsonish(x):
    if isinstance(x, str):
        try:
            return json.loads(x)
        except Exception:
            return x
    return x

def _canon_label(x):
    s = str(x).strip().lower()
    try:
        v = float(s)
        if math.isfinite(v) and v.is_integer():
            return str(int(v))
    except Exception:
        pass
    return s

def _gold_metadata(q, gold_answer):
    if not isinstance(gold_answer, dict):
        raise ValueError("missing gold answer")
    probs_raw = _jsonish(gold_answer.get("probabilities", {}))
    if not isinstance(probs_raw, dict) or len(probs_raw) < 2:
        raise ValueError("gold probabilities missing or malformed")

    gold_keys = [str(k) for k in probs_raw.keys()]
    if q["t"] == "choice":
        labels = [str(k) for k in q["crit"].keys()]
    elif q["t"] == "noul":
        labels = ["false", "true"]
    else:
        labels = list(gold_keys)
        if all(_canon_label(k).lstrip("-").isdigit() for k in labels):
            labels = sorted(labels, key=lambda k: int(_canon_label(k)))

    lookup = {_canon_label(k): float(v) for k, v in probs_raw.items()}
    if len(labels) != len(gold_keys) or any(_canon_label(k) not in lookup for k in labels):
        raise ValueError("Gold labels do not match encoded option order")

    gold_probs = np.asarray([lookup[_canon_label(k)] for k in labels], dtype=np.float64)
    gold_probs = np.clip(gold_probs, 0.0, None)
    if not np.isfinite(gold_probs).all() or gold_probs.sum() <= 0:
        raise ValueError("invalid gold probability vector")
    gold_probs /= gold_probs.sum()

    gold_label = _canon_label(gold_answer.get("label", labels[int(gold_probs.argmax())]))
    canon_labels = [_canon_label(k) for k in labels]
    if gold_label not in canon_labels:
        raise ValueError(f"gold label {gold_label!r} not in {labels!r}")
    return labels, gold_probs.tolist(), canon_labels.index(gold_label)

def make_items_from_rows(rows, seed, shuffle_items):
    out = []
    head_max_len = int(source_cfg.get("head_max_len", 192))
    for row in rows:
        state = _jsonish(row["state"])
        questions = _jsonish(row["questions"])
        gold = _jsonish(row["gold"])
        case_id = str(row.get("id", ""))
        workflow = str(row.get("workflow", "unknown"))
        state_text = state if isinstance(state, str) else json.dumps(state, ensure_ascii=False)
        state_ids = tokenizer(
            state_text.replace(tokenizer.mask_token, " "),
            add_special_tokens=False,
        )["input_ids"]

        for qid, qdef in questions.items():
            q = normalize_question(qdef)
            ids, markers = build_sequence(
                tokenizer,
                state,
                q,
                max_len=min(MAX_LEN, int(source_cfg.get("max_len", MAX_LEN))),
                head_max_len=head_max_len,
                truncate_left=isinstance(state, list),
                state_ids=state_ids,
            )
            if len(markers) < 2:
                continue
            labels, gold_probs, gold_index = _gold_metadata(q, gold[qid])
            if len(labels) != len(markers):
                raise ValueError(
                    f"{case_id}/{qid}: {len(markers)} markers vs {len(labels)} gold labels"
                )
            out.append({
                "ids": ids,
                "markers": markers,
                "qtype": QTYPES[q["t"]],
                "qtype_name": q["t"],
                "qid": str(qid),
                "case_id": case_id,
                "workflow": workflow,
                "labels": labels,
                "gold_probs": gold_probs,
                "gold_index": int(gold_index),
            })
    if shuffle_items:
        random.Random(seed).shuffle(out)
    return out

# Preserve V5.0's exact case-level TRAIN and PROBE split to reproduce its
# 120/120 step-120 finding. Use the DATASET'S DISTINCT TEST SPLIT for AUDIT.
# Reuse historical V5.1 train/PROBE for comparability, but SELECT using
# unused second-half DEV cases. TEST split stays unseen until after freezing.
rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="train"))
random.Random(SEED).shuffle(rows)
assert TRAIN_CASES+DEV_CASES<=len(rows)
train_rows=rows[:TRAIN_CASES]
historical_probe_rows=rows[TRAIN_CASES:TRAIN_CASES+DEV_CASES//2]
gold_val_rows=rows[TRAIN_CASES+DEV_CASES//2:TRAIN_CASES+DEV_CASES]
train_items=make_items_from_rows(train_rows,SEED,True)
probe_items=make_items_from_rows(historical_probe_rows,SEED+1,True)
gold_val_items=make_items_from_rows(gold_val_rows,SEED+2,True)
# Case namespaces protect against overlapping local split ids.
test_rows=list(load_dataset("LocalLLaMA/typed-decisions","all",split="test"))
random.Random(SEED+9001).shuffle(test_rows)
audit_rows=[dict(row,id="TEST::"+str(row.get("id",i)))
            for i,row in enumerate(test_rows)]
audit_items=make_items_from_rows(audit_rows,SEED+9001,True)
assert len(probe_items)>=HISTORICAL_PROBE_DECISIONS
assert len(gold_val_items)>=GOLD_VAL_DECISIONS
assert len(audit_items)>=AUDIT_DECISIONS
groups=[{x["case_id"] for x in seq} for seq in
        (train_items,probe_items,gold_val_items,audit_items)]
for i in range(len(groups)):
    for j in range(i+1,len(groups)):
        assert groups[i].isdisjoint(groups[j]),(i,j)
def direct_batch(selected):
    b=collate_items(selected,int(tokenizer.pad_token_id))
    return {k:v.to(device) for k,v in b.items()}
def batch(items,offset,n=BATCH):
    return direct_batch([items[(offset+i)%len(items)] for i in range(n)])
print("TRAIN:",len(train_items),"OLD PROBE:",len(probe_items),
      "NEW GOLD VAL:",len(gold_val_items),"FINAL TEST:",len(audit_items))


README.md: 0.00B [00:00, ?B/s]

all/train-00000-of-00001.parquet:   0%|          | 0.00/599k [00:00<?, ?B/s]

all/test-00000-of-00001.parquet:   0%|          | 0.00/222k [00:00<?, ?B/s]

Generating train split:   0%|          | 0/1200 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/400 [00:00<?, ? examples/s]

TRAIN: 2500 OLD PROBE: 600 NEW GOLD VAL: 600 FINAL TEST: 2000


In [5]:
#@title 5. Accuracy, divergence and paired CUDA/wall timing
@torch.inference_mode()
def probs_for(model,b):
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=model(**b)
    return torch.softmax(scores.float().masked_fill(
        ~b["marker_mask"].bool(),-1e9),dim=-1).cpu()

@torch.inference_mode()
def get_reference(items,limit):
    out=[]
    for start in range(0,min(limit,len(items)),BATCH):
        chosen=items[start:min(start+BATCH,limit)]
        p=probs_for(teacher,direct_batch(chosen))
        for j,item in enumerate(chosen):
            out.append(dict(probs=p[j],teacher=int(p[j].argmax()),
                gold=int(item["gold_index"]),case=item["case_id"],qid=item["qid"]))
    return out

@torch.inference_mode()
def quality(model,items,ref):
    agree=correct=tcorrect=0;js=0.
    n=len(ref)
    for start in range(0,n,BATCH):
        chosen=items[start:min(n,start+BATCH)]
        p=probs_for(model,direct_batch(chosen))
        for j,item in enumerate(chosen):
            t=ref[start+j]["probs"]; s=p[j]
            assert s.shape==t.shape
            m=(s+t)/2
            js+=float(.5*((s*(s.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()+
                            (t*(t.clamp_min(1e-8).log()-m.clamp_min(1e-8).log())).sum()))
            guess=int(s.argmax())
            agree+=int(guess==ref[start+j]["teacher"])
            correct+=int(guess==item["gold_index"])
            tcorrect+=int(ref[start+j]["teacher"]==item["gold_index"])
    return dict(decisions=n,teacher_agreement=agree/n,gold_accuracy=correct/n,
        teacher_gold_accuracy=tcorrect/n,gold_drop=(tcorrect-correct)/n,mean_js=js/n)

BENCH_BATCHES=[batch(probe_items,0,BATCH),batch(probe_items,11,BATCH)]
@torch.inference_mode()
def benchmark_paired(reference,candidate,repeats=BENCH_REPEATS):
    """Alternating A/B/B/A with CUDA events AND synchronized wall-clock timings.
    All timings exclude Colab compilation, dataset loading and model downloading."""
    results=[]
    for j,b in enumerate(BENCH_BATCHES):
        def once(m):
            torch.cuda.synchronize()
            a=torch.cuda.Event(enable_timing=True)
            z=torch.cuda.Event(enable_timing=True)
            before=time.perf_counter()
            a.record()
            with torch.autocast("cuda",dtype=amp_dtype):
                outputs=m(**b)
            z.record();z.synchronize()
            wall=(time.perf_counter()-before)*1000
            return float(a.elapsed_time(z)),float(wall)
        for k in range(4):
            once(reference);once(candidate)
        d={"reference":[],"candidate":[]}
        w={"reference":[],"candidate":[]}
        for k in range(repeats):
            order=("reference","candidate","candidate","reference") if k%2==0 else (
                "candidate","reference","reference","candidate")
            for name in order:
                gpu,cpu=once(reference if name=="reference" else candidate)
                d[name].append(gpu);w[name].append(cpu)
        ref=float(np.median(d["reference"]))
        cand=float(np.median(d["candidate"]))
        rw=float(np.median(w["reference"]))
        cw=float(np.median(w["candidate"]))
        results.append(dict(shape=j,batch=int(b["input_ids"].shape[0]),
            tokens=int(b["input_ids"].shape[1]),
            native_gpu_ms=ref,candidate_gpu_ms=cand,gpu_speedup=ref/max(cand,1e-9),
            native_wall_ms=rw,candidate_wall_ms=cw,wall_speedup=rw/max(cw,1e-9),
            native_p95_ms=float(np.percentile(d["reference"],95)),
            candidate_p95_ms=float(np.percentile(d["candidate"],95)),n=len(d["reference"])))
    return dict(median_gpu_speedup=float(np.median([r["gpu_speedup"] for r in results])),
        min_gpu_speedup=float(min(r["gpu_speedup"] for r in results)),
        median_wall_speedup=float(np.median([r["wall_speedup"] for r in results])),
        min_wall_speedup=float(min(r["wall_speedup"] for r in results)),
        per_shape=results)

def valid_quality(q):
    return (q["teacher_agreement"]>=MIN_TEACHER_AGREEMENT and
            q["gold_drop"]<=MAX_GOLD_DROP_PP/100 and q["mean_js"]<=MAX_JS)
def valid_speed(s):
    return (s["min_gpu_speedup"]>=MIN_GPU_SPEED_RATIO and
            s["min_wall_speedup"]>=MIN_WALL_SPEED_RATIO)



In [6]:
#@title 6. Reconstruct compressed control; optionally restore V5.2 checkpoint
# This exact ordered list comes from the user's V5.0 greedy_combinations.json.
# It replaces the 520-candidate rank sweep; all unchanged projections are
# restored from PINNED pretrained V4.6 weights.
SELECTED_SPECS = [
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wqkv",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 26,
    "leader": 25,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 64
  },
  {
    "layer": 17,
    "leader": 16,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 20,
    "leader": 19,
    "block": "attn",
    "component": "Wo",
    "mode": "pair_delta",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wqkv",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 27,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 32
  },
  {
    "layer": 23,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 25,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 24,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 64
  },
  {
    "layer": 26,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 21,
    "block": "mlp",
    "component": "Wi",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 20,
    "block": "mlp",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 8,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 3,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 2,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  },
  {
    "layer": 15,
    "block": "attn",
    "component": "Wo",
    "mode": "own_svd",
    "rank": 256
  }
]
assert len(SELECTED_SPECS)==20
assert len({(s["layer"],s["block"],s["component"]) for s in SELECTED_SPECS})==20
assert sum(s["mode"]=="pair_delta" for s in SELECTED_SPECS)==4
# Stable fingerprint of the exact 20 selected projections and ranks.
# Used in model config, provenance and release verification.
spec_hash=hashlib.sha256(
    json.dumps(SELECTED_SPECS,sort_keys=True,separators=(",",":")).encode("utf-8")
).hexdigest()
assert len(spec_hash)==64
print("Compression specification SHA-256:",spec_hash)
def family_key(f):
    return (int(f["layer"]),str(f["block"]),str(f["component"]),str(f["mode"]))
def deterministic_factors(f):
    family_seed=SEED+sum((i+1)*ord(ch) for i,ch in enumerate(
        ":".join(map(str,family_key(f)))))
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(family_seed)
        if f["mode"]=="own_svd":
            return v53.own_svd_factors(
                v53.projection_at(student,f),
                max_rank=256,oversampling=8,niter=2)[:2]
        return v53.svd_factors(
            student,int(f["leader"]),int(f["layer"]),
            str(f["component"]),max_rank=256,oversampling=8,niter=2)[:2]

PROBE_REF=get_reference(probe_items,HISTORICAL_PROBE_DECISIONS)
GOLD_VAL_REF=get_reference(gold_val_items,GOLD_VAL_DECISIONS)

native_q=quality(student,probe_items,PROBE_REF)
assert native_q["teacher_agreement"]==1.0,native_q
for j,s in enumerate(SELECTED_SPECS,1):
    factors=deterministic_factors(s)
    old=v53.replace_projection(student,s,factors=factors,initialized=True)
    assert v53.projection_at(student,s) is not old
    print(f"Selected {j:02d}/{len(SELECTED_SPECS)}: layer={s['layer']} "
          f"{s['block']}.{s['component']} {s['mode']} rank={s['rank']}")
unique_student=v53.unique_parameter_count(student)
estimated_bytes=(native_unique-unique_student)*4
assert estimated_bytes>0
initial_probe=quality(student,probe_items,PROBE_REF)
initial_gold_validation=quality(student,gold_val_items,GOLD_VAL_REF)
print("Historical initial PROBE (not used for selection):",initial_probe)
print("New gold-validation baseline:",initial_gold_validation)
print("Teacher gold accuracy in gold-validation:",initial_gold_validation["teacher_gold_accuracy"])

print("Projected parameter savings:",f"{estimated_bytes:,} bytes",
      f"({estimated_bytes/1e6:.3f} MB)")
(OUT/"selected_projections.json").write_text(json.dumps({
    "source_repo":TEACHER_REPO,"source_revision":PINNED_TEACHER_REVISION,
    "specifications":SELECTED_SPECS,"estimated_saved_fp32_bytes":estimated_bytes,
    "initial_probe":initial_probe},indent=2),encoding="utf-8")

# Optional exact, already-gold-tuned V5.2 standalone checkpoint. The config
# must contain the SAME 20 selected projections in the SAME order.
BASELINE_SOURCE="reconstructed_V5.1_frozen_head"
if BASELINE_V52_STANDALONE_DIR.strip():
    root=Path(BASELINE_V52_STANDALONE_DIR.strip()).resolve()
    cfg=json.loads((root/"config.json").read_text(encoding="utf-8"))
    assert cfg["compressed_projections"]==SELECTED_SPECS
    assert cfg["source_revision"]==PINNED_TEACHER_REVISION
    tensors=load_file(str(root/"model.safetensors"),device="cpu")
    student.load_state_dict(tensors,strict=True)
    BASELINE_SOURCE=str(root)
student.eval().requires_grad_(False)
BASELINE_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
BASELINE_OLD_PROBE=quality(student,probe_items,PROBE_REF)
print("CONTROL BASELINE SOURCE:",BASELINE_SOURCE)
print("CONTROL GOLD VALIDATION:",BASELINE_VAL)
(OUT/"baseline_metrics.json").write_text(json.dumps(dict(
    source=BASELINE_SOURCE,teacher_revision=PINNED_TEACHER_REVISION,
    gold_validation=BASELINE_VAL,historical_probe=BASELINE_OLD_PROBE,
    compressed_spec_hash=spec_hash),indent=2))


Compression specification SHA-256: 5cb1804e2e2bea0180c19f3179f3075c9be467e49cc3ac99257f3318f4c6ac6e
Selected 01/20: layer=26 attn.Wqkv pair_delta rank=64
Selected 02/20: layer=26 attn.Wo pair_delta rank=64
Selected 03/20: layer=17 attn.Wo pair_delta rank=256
Selected 04/20: layer=20 attn.Wo pair_delta rank=256
Selected 05/20: layer=27 attn.Wqkv own_svd rank=64
Selected 06/20: layer=24 mlp.Wi own_svd rank=256
Selected 07/20: layer=23 attn.Wqkv own_svd rank=256
Selected 08/20: layer=27 attn.Wo own_svd rank=32
Selected 09/20: layer=23 attn.Wo own_svd rank=32
Selected 10/20: layer=23 mlp.Wo own_svd rank=256
Selected 11/20: layer=25 mlp.Wo own_svd rank=256
Selected 12/20: layer=24 mlp.Wo own_svd rank=64
Selected 13/20: layer=26 mlp.Wo own_svd rank=256
Selected 14/20: layer=21 mlp.Wo own_svd rank=256
Selected 15/20: layer=21 mlp.Wi own_svd rank=256
Selected 16/20: layer=20 mlp.Wo own_svd rank=256
Selected 17/20: layer=8 attn.Wo own_svd rank=256
Selected 18/20: layer=3 attn.Wo own_svd rank=25

659

In [7]:
#@title 7. Screen liquid CeNN replacement candidates using REAL FFN activations
# All scan scores are from the GOLD VALIDATION subset, never TEST.
# The direct-replacement ablation is trained locally to imitate each frozen
# FFN output, which is dramatically cheaper than distilling the entire model.
# Candidate FEASIBILITY is validated before the full gold-supervised phase.
ALL_LAYERS=tuple(range(28)) if SCAN_ALL_28_LAYERS else tuple(CANDIDATE_LAYERS)
assert len(ALL_LAYERS)>0 and len(set(ALL_LAYERS))==len(ALL_LAYERS)
assert all(0<=int(i)<28 for i in ALL_LAYERS)
SCREEN_REF=GOLD_VAL_REF[:SCREEN_VAL_DECISIONS]
SCREEN_ITEMS=gold_val_items[:SCREEN_VAL_DECISIONS]
BASELINE_SCREEN=quality(student,SCREEN_ITEMS,SCREEN_REF)
scan=[]
stored_candidates={}
pretraining_diagnostics=[]
def capture_projection_io(layer):
    saved=[]
    target=student.encoder.layers[int(layer)].mlp.Wo
    def capture(_module,args,output):
        saved.append((args[0].detach().to("cpu",dtype=torch.float16).clone(),
                     output.detach().to("cpu",dtype=torch.float16).clone()))
    hook=target.register_forward_hook(capture)
    try:
        for k in range(CAPTURE_BATCHES):
            b=batch(train_items,k*BATCH,BATCH)
            # Never save inference-mode activations, which cannot be used
            # for weight-gradient backward in a subsequent training step.
            with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
                student(**b)
    finally:
        hook.remove()
    assert len(saved)==CAPTURE_BATCHES,(layer,len(saved))
    return target,saved

for layer in ALL_LAYERS:
    old,samples=capture_projection_io(layer)
    candidate=v53.LiquidCeNNProjection(
        old,width=CE_WIDTH,steps=CE_STEPS,mode="replace",
        dt_bound=CE_DT_BOUND).to(device=DEVICE,dtype=torch.float32)
    optimizer_local=torch.optim.AdamW(candidate.parameters(),
                                      lr=IMITATION_LR,weight_decay=0.)
    loss_trace=[]
    candidate.train()
    for k in range(IMITATION_STEPS):
        xs,ys=samples[k%len(samples)]
        x=xs.to(DEVICE,dtype=torch.float32)
        y=ys.to(DEVICE,dtype=torch.float32)
        optimizer_local.zero_grad(set_to_none=True)
        output=candidate(x)
        loss=F.mse_loss(output,y) / y.float().square().mean().clamp_min(1e-6)
        assert bool(torch.isfinite(loss)),f"NaN in local CeNN MSE layer={layer}"
        loss.backward()
        torch.nn.utils.clip_grad_norm_(candidate.parameters(),1.)
        optimizer_local.step()
        if k==0 or (k+1)%20==0 or k+1==IMITATION_STEPS:
            loss_trace.append(dict(step=k+1,relative_mse=float(loss.detach().cpu())))
    candidate.eval().requires_grad_(False)
    # Direct-replacement quality on small VALIDATION set only.
    student.encoder.layers[int(layer)].mlp.Wo=candidate
    try:
        tested=quality(student,SCREEN_ITEMS,SCREEN_REF)
    finally:
        student.encoder.layers[int(layer)].mlp.Wo=old
    last_mse=loss_trace[-1]["relative_mse"] if loss_trace else None
    improvement_pp=100*(tested["gold_accuracy"]-BASELINE_SCREEN["gold_accuracy"])
    report=dict(layer=int(layer),mode="replace",width=CE_WIDTH,steps=CE_STEPS,
        imitation_steps=IMITATION_STEPS,relative_mse=last_mse,
        screen_quality=tested,gold_delta_vs_compressed_baseline_pp=improvement_pp,
        original_projection_parameters=sum(p.numel() for p in old.parameters()),
        cenn_parameters=sum(p.numel() for p in candidate.parameters()),
        loss_trace=loss_trace)
    # Replacements must be close to baseline on gold correctness AND decision
    # distribution before they can be promoted for the full experiment.
    report["replacement_screen_pass"]=(
        tested["gold_accuracy"]>=BASELINE_SCREEN["gold_accuracy"]-0.025 and
        tested["teacher_agreement"]>=0.90 and tested["mean_js"]<=0.020)
    scan.append(report)
    stored_candidates[int(layer)]=OrderedDict(
        (name,t.detach().cpu().contiguous().clone())
        for name,t in candidate.state_dict().items())
    print(f"CeNN replace layer={layer:02d}, relativeMSE={last_mse}, "
          f"gold={tested['gold_accuracy']:.3f}, "
          f"teacherAgree={tested['teacher_agreement']:.3f}, "
          f"passed={report['replacement_screen_pass']}")
    (OUT/"liquid_cenn_screen.json").write_text(json.dumps(
       dict(baseline_screen=BASELINE_SCREEN,
            source=BASELINE_SOURCE,scan=scan),indent=2))
    del optimizer_local,candidate,samples
    gc.collect()
# Prefer a validated replacement; otherwise residual intervention preserves
# all previous projections. This selection uses VALIDATION only.
passing=[x for x in scan if x["replacement_screen_pass"]]
if passing:
    chosen=max(passing,key=lambda x:(
      x["screen_quality"]["gold_accuracy"],-x["screen_quality"]["mean_js"],
      -x["relative_mse"]))
    SELECTED_MODE="replace"
else:
    chosen=min(scan,key=lambda x:(
      float("inf") if x["relative_mse"] is None else x["relative_mse"],
      -x["screen_quality"]["gold_accuracy"]))
    SELECTED_MODE="residual"
SELECTED_LAYER=int(chosen["layer"])
original_ffn=v53.install_liquid_ffn(
    student,SELECTED_LAYER,width=CE_WIDTH,steps=CE_STEPS,
    mode=SELECTED_MODE,dt_bound=CE_DT_BOUND)
liquid_cell=student.encoder.layers[SELECTED_LAYER].mlp.Wo
trained_weights=stored_candidates[SELECTED_LAYER]
with torch.no_grad():
    # Base module weights stay intact in residual mode; only CeNN-specific
    # matrices copied from replacement pretraining. Scale remains zero.
    for name,parameter in liquid_cell.named_parameters():
        if name in trained_weights:
            parameter.copy_(trained_weights[name].to(
                device=parameter.device,dtype=parameter.dtype))
del stored_candidates
gc.collect()
pretrain_baseline=quality(student,gold_val_items,GOLD_VAL_REF)
if SELECTED_MODE=="residual":
    assert abs(pretrain_baseline["mean_js"]-BASELINE_VAL["mean_js"])<1e-7
    assert pretrain_baseline["gold_accuracy"]==BASELINE_VAL["gold_accuracy"]
print("SELECTED LIQUID-CENN MODE:",SELECTED_MODE,"LAYER:",SELECTED_LAYER)
print("Initial liquid gold validation:",pretrain_baseline)
LIQUID_CONFIG=dict(layer=SELECTED_LAYER,mode=SELECTED_MODE,
    width=CE_WIDTH,steps=CE_STEPS,dt_bound=CE_DT_BOUND)
(OUT/"selected_liquid_architecture.json").write_text(json.dumps(dict(
    liquid_cell=LIQUID_CONFIG,selected_scan_result=chosen,
    control_baseline=BASELINE_VAL,initial_liquid_quality=pretrain_baseline),
    indent=2))


CeNN replace layer=08, relativeMSE=0.7233724594116211, gold=0.833, teacherAgree=0.938, passed=True
CeNN replace layer=17, relativeMSE=0.7579463720321655, gold=0.833, teacherAgree=0.938, passed=True
CeNN replace layer=20, relativeMSE=0.7682027816772461, gold=0.854, teacherAgree=1.000, passed=True
CeNN replace layer=23, relativeMSE=0.7595962285995483, gold=0.854, teacherAgree=1.000, passed=True
CeNN replace layer=24, relativeMSE=0.5902574062347412, gold=0.854, teacherAgree=1.000, passed=True
CeNN replace layer=26, relativeMSE=0.7641845941543579, gold=0.854, teacherAgree=1.000, passed=True
SELECTED LIQUID-CENN MODE: replace LAYER: 24
Initial liquid gold validation: {'decisions': 180, 'teacher_agreement': 0.9777777777777777, 'gold_accuracy': 0.8333333333333334, 'teacher_gold_accuracy': 0.85, 'gold_drop': 0.016666666666666666, 'mean_js': 0.002800363735797065}


1621

In [8]:
#@title 8. Fine-tune the Liquid CeNN on gold labels, select validation checkpoint
# All 28 encoder layers remain present; no original pretrained weight updates.
student.requires_grad_(False)
liquid_cell=student.encoder.layers[SELECTED_LAYER].mlp.Wo
for name,p in liquid_cell.named_parameters():
    # Residual base FFN is FROZEN; only liquid dynamics are trainable.
    if not name.startswith("base."):
        p.requires_grad_(True)
if TRAIN_DECISION_SCORER:
    for p in student.scorer.parameters(): p.requires_grad_(True)
trainable=[p for p in student.parameters() if p.requires_grad]
assert trainable
assert not any(p.requires_grad for p in teacher.parameters())
trainable_specs={name:list(p.shape)
      for name,p in student.named_parameters() if p.requires_grad}
trainable_names=set(trainable_specs)
trainable_fingerprint=hashlib.sha256(json.dumps(
       trainable_specs,sort_keys=True).encode()).hexdigest()
student.eval()
BEST=OUT/"best_liquid_trainable.safetensors"
HISTORY=OUT/"liquid_training_progress.json"
def save_selected(step,quality_value,reason):
    from collections import OrderedDict
    state=OrderedDict((name,p.detach().cpu().contiguous().clone())
      for name,p in student.named_parameters() if p.requires_grad)
    save_file(state,str(BEST),metadata={
      "format":"pt","fingerprint":trainable_fingerprint,"step":str(step)})
    (OUT/"best_liquid_trainable.json").write_text(json.dumps(
       dict(step=step,quality=quality_value,reason=reason,
            trainable_fingerprint=trainable_fingerprint,
            liquid_config=LIQUID_CONFIG),indent=2))
def restore_selected():
    weights=load_file(str(BEST),device="cpu")
    assert set(weights)==trainable_names
    with torch.no_grad():
        for name,p in student.named_parameters():
            if name in weights:
                p.copy_(weights[name].to(p.device,p.dtype))
def rank_gold(q):
    return (q["gold_accuracy"],
            -max(0.,q["teacher_gold_accuracy"]-q["gold_accuracy"]),
            -q["mean_js"])
def make_gold_batch(step):
    entries=[train_items[((step-1)*TRAIN_B+j)%len(train_items)]
             for j in range(TRAIN_B)]
    b=direct_batch(entries)
    assert bool((b["marker_pos"][b["marker_mask"]]<TRAIN_MAX_TOKENS).all())
    for key in ("input_ids","attention_mask"):
        b[key]=b[key][:,:TRAIN_MAX_TOKENS]
    gold=torch.tensor([item["gold_index"] for item in entries],
        dtype=torch.long,device=DEVICE)
    soft=torch.zeros(b["marker_mask"].shape,dtype=torch.float32,device=DEVICE)
    for i,item in enumerate(entries):
        goldp=torch.tensor(item["gold_probs"],dtype=torch.float32,device=DEVICE)
        soft[i,:len(goldp)]=goldp/goldp.sum().clamp_min(1e-8)
    return b,gold,soft
best_score=quality(student,gold_val_items,GOLD_VAL_REF)
best_step=0
save_selected(0,best_score,"start_from_imitation_or_identity")
training_history=[]
optimizer=torch.optim.AdamW(trainable,lr=TRAIN_LR,weight_decay=0.)
scaler=torch.amp.GradScaler("cuda",enabled=amp_dtype==torch.float16)
patience_checks=0
for step in range(1,TRAIN_STEPS+1):
    b,gold,soft=make_gold_batch(step)
    with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
        teacher_logits,_=teacher(**b)
    optimizer.zero_grad(set_to_none=True)
    with torch.autocast("cuda",dtype=amp_dtype):
        scores,_=student(**b)
    valid=b["marker_mask"].bool()
    s=scores.float().masked_fill(~valid,-1e9)
    t=teacher_logits.float().masked_fill(~valid,-1e9)
    logs=F.log_softmax(s,dim=-1)
    ce=F.nll_loss(logs,gold)
    soft_ce=-(soft*logs).sum(-1).mean()
    kl=F.kl_div(F.log_softmax(s/2.,dim=-1),
       F.softmax(t/2.,dim=-1),reduction="batchmean")*4.
    loss=GOLD_CE_WEIGHT*ce+GOLD_SOFT_WEIGHT*soft_ce+TEACHER_KL_WEIGHT*kl
    if not bool(torch.isfinite(loss)):
        raise RuntimeError(f"Non-finite CeNN training loss at {step}")
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(trainable,1.)
    scaler.step(optimizer);scaler.update()
    if step%TRAIN_CHECK_EVERY==0 or step==TRAIN_STEPS:
        q=quality(student,gold_val_items,GOLD_VAL_REF)
        improved=rank_gold(q)>rank_gold(best_score)
        if improved:
            best_score=q;best_step=step
            save_selected(step,q,"best_gold_validation")
            patience_checks=0
        else:
            patience_checks+=1
        training_history.append(dict(step=step,
              loss=float(loss.detach().cpu()),quality=q,
              gold_gain_vs_teacher_pp=100*(
                  q["gold_accuracy"]-q["teacher_gold_accuracy"]),
              improved=improved))
        HISTORY.write_text(json.dumps(training_history,indent=2))
        print(f"step={step} mode={SELECTED_MODE} "
             f"gold={q['gold_accuracy']:.3%} "
             f"teacher={q['teacher_gold_accuracy']:.3%} "
             f"JS={q['mean_js']:.6f} selected={improved}")
        if patience_checks>=5:
            print("EARLY STOP: 5 validation checks without improved gold correctness")
            break
restore_selected()
student.eval().requires_grad_(False)
FINAL_VAL=quality(student,gold_val_items,GOLD_VAL_REF)
HISTORICAL_PROBE=quality(student,probe_items,PROBE_REF)
assert FINAL_VAL["gold_accuracy"]==best_score["gold_accuracy"]
(OUT/"liquid_training_summary.json").write_text(json.dumps(dict(
    liquid_config=LIQUID_CONFIG,control_validation=BASELINE_VAL,
    selected_step=best_step,final_validation=FINAL_VAL,
    historical_probe=HISTORICAL_PROBE,
    training_history=training_history,
    trainable_parameter_count=sum(p.numel() for p in trainable)
),indent=2))
print("CeNN validation:",FINAL_VAL,"selected step:",best_step)


step=20 mode=replace gold=83.333% teacher=85.000% JS=0.002742 selected=True
step=40 mode=replace gold=83.333% teacher=85.000% JS=0.002750 selected=False
step=60 mode=replace gold=83.333% teacher=85.000% JS=0.002741 selected=True
step=80 mode=replace gold=83.333% teacher=85.000% JS=0.002709 selected=True
step=100 mode=replace gold=83.333% teacher=85.000% JS=0.002674 selected=True
step=120 mode=replace gold=83.333% teacher=85.000% JS=0.002632 selected=True
step=140 mode=replace gold=83.333% teacher=85.000% JS=0.002586 selected=True
step=160 mode=replace gold=82.778% teacher=85.000% JS=0.002566 selected=False
step=180 mode=replace gold=82.778% teacher=85.000% JS=0.002513 selected=False
step=200 mode=replace gold=82.778% teacher=85.000% JS=0.002469 selected=False
CeNN validation: {'decisions': 180, 'teacher_agreement': 0.9777777777777777, 'gold_accuracy': 0.8333333333333334, 'teacher_gold_accuracy': 0.85, 'gold_drop': 0.016666666666666666, 'mean_js': 0.0025860953161302154} selected step: 1

In [9]:
#@title 9. FROZEN one-time TEST audit, gold error analysis and timing
assert not any(p.requires_grad for p in student.parameters())
# Build a separate compressed reference without CeNN for valid AB comparison,
# using the unchanged full weights from V5.0/V5.2 baseline.
comparison_baseline,_=v53.load_model(parent_dir,device="cuda")
comparison_baseline.encoder.config.reference_compile=False
comparison_baseline.eval().requires_grad_(False)
for spec in SELECTED_SPECS:
    # Factor recreation uses the new model, not the liquid-wrapped student.
    # Reconstruct original pretrained SVD factors with the exact V5.0
    # per-projection RNG seed used in the baseline and liquid experiments.
    label=":".join(map(str,(int(spec["layer"]),spec["block"],
                           spec["component"],spec["mode"])))
    family_seed=SEED+sum((i+1)*ord(ch) for i,ch in enumerate(label))
    with torch.random.fork_rng(devices=[torch.cuda.current_device()]):
        torch.manual_seed(family_seed)
        if spec["mode"]=="own_svd":
            A,B=v53.own_svd_factors(
                v53.projection_at(comparison_baseline,spec),
                max_rank=256,oversampling=8,niter=2)[:2]
        else:
            A,B=v53.svd_factors(comparison_baseline,int(spec["leader"]),
                int(spec["layer"]),spec["component"],
                max_rank=256,oversampling=8,niter=2)[:2]
    v53.replace_projection(comparison_baseline,spec,factors=(A,B))
if BASELINE_V52_STANDALONE_DIR.strip():
    d=Path(BASELINE_V52_STANDALONE_DIR.strip()).resolve()
    state=load_file(str(d/"model.safetensors"),device="cpu")
    comparison_baseline.load_state_dict(state,strict=True)
comparison_baseline.eval().requires_grad_(False)
CONTROL_BENCH_BATCHES=BENCH_BATCHES
BENCH_BATCHES=[batch(audit_items,0,BATCH),batch(audit_items,11,BATCH)]
TEST_REF=get_reference(audit_items,AUDIT_DECISIONS)
test_quality=quality(student,audit_items,TEST_REF)
baseline_test=quality(comparison_baseline,audit_items,TEST_REF)
speed_against_baseline=benchmark_paired(
    comparison_baseline,student,repeats=max(BENCH_REPEATS,15))
speed_against_teacher=benchmark_paired(
    teacher,student,repeats=max(BENCH_REPEATS,15))
BENCH_BATCHES=CONTROL_BENCH_BATCHES
@torch.inference_mode()
def disagreement_report(items,ref):
    from collections import Counter
    count=Counter()
    by_kind={}
    for i in range(0,len(ref),BATCH):
        records=items[i:i+BATCH]
        p=probs_for(student,direct_batch(records))
        for j,item in enumerate(records):
            actual=int(p[j].argmax())
            truth=int(item["gold_index"])
            source=int(ref[i+j]["teacher"])
            if source!=truth and actual==truth: label="corrected_teacher_error"
            elif source==truth and actual!=truth: label="introduced_error"
            elif source==truth: label="both_correct"
            else: label="both_wrong"
            count[label]+=1
            by_kind.setdefault(item["qtype_name"],Counter())[label]+=1
    return dict(overall=dict(count),
        by_type={k:dict(v) for k,v in by_kind.items()})
test_errors=disagreement_report(audit_items,TEST_REF)
report=dict(teacher_gold=test_quality["teacher_gold_accuracy"],
    baseline_gold=baseline_test["gold_accuracy"],
    liquid_gold=test_quality["gold_accuracy"],
    gold_difference_pp=100*(test_quality["gold_accuracy"]-
        test_quality["teacher_gold_accuracy"]),
    gold_gain_vs_baseline_pp=100*(test_quality["gold_accuracy"]-
        baseline_test["gold_accuracy"]),
    test_metrics=test_quality,baseline_test_metrics=baseline_test,
    error_breakdown=test_errors,
    gpu_wall_speed_vs_baseline=speed_against_baseline,
    gpu_wall_speed_vs_teacher=speed_against_teacher)
(OUT/"liquid_test_report.json").write_text(json.dumps(report,indent=2))
print("TEACHER GOLD:",report["teacher_gold"],"BASE GOLD:",report["baseline_gold"],
      "LIQUID GOLD:",report["liquid_gold"])
print("LIQUID vs baseline:",report["gold_gain_vs_baseline_pp"],"pp")
print("LIQUID vs teacher:",report["gold_difference_pp"],"pp")
print("GPU ratios:",speed_against_baseline["median_gpu_speedup"])


TEACHER GOLD: 0.765 BASE GOLD: 0.755 LIQUID GOLD: 0.755
LIQUID vs baseline: 0.0 pp
LIQUID vs teacher: -1.0000000000000009 pp
GPU ratios: 0.9890930899200465


In [10]:
#@title 10. Export STRICT standalone V5.3 model and verify reload logits
from collections import OrderedDict
(DEST/"encoder").mkdir(parents=True,exist_ok=True)
(DEST/"tokenizer").mkdir(parents=True,exist_ok=True)
student.encoder.config.save_pretrained(str(DEST/"encoder"))
tokenizer.save_pretrained(str(DEST/"tokenizer"))
(DEST/"modeling_laya_v53.py").write_text(V53_RUNTIME_CODE,encoding="utf-8")
cfg=dict(format="laya-v53-liquid-cenn-v1",model_type="laya-v53-liquid-cenn",
    head_context_budget=64,head_layers=int(source_config["head_layers"]),
    source_repo=TEACHER_REPO,source_revision=PINNED_TEACHER_REVISION,
    compressed_projections=SELECTED_SPECS,liquid_cell=LIQUID_CONFIG,
    selected_training_step=best_step,baseline_source=BASELINE_SOURCE,
    output_scope="typed-decision option scores only")
(DEST/"config.json").write_text(json.dumps(cfg,indent=2))
state=student.state_dict()
assert not any("._shared." in k for k in state)
weight_state=OrderedDict((k,v.detach().cpu().contiguous().clone())
   for k,v in state.items())
save_file(weight_state,str(DEST/"model.safetensors"),
     metadata={"format":"pt","architecture":"laya-v53-liquid-cenn-v1"})
del state,weight_state;gc.collect()
model_bytes=(DEST/"model.safetensors").stat().st_size
baseline_bytes=(parent_dir/"model.safetensors").stat().st_size
actual_saved_bytes=baseline_bytes-model_bytes
reloaded,rtok=v53.load_v53_model(DEST,device="cuda")
smoke=batch(gold_val_items,0,BATCH)
with torch.no_grad(),torch.autocast("cuda",dtype=amp_dtype):
    aa,_=student(**smoke)
    bb,_=reloaded(**smoke)
max_abs_diff=float((aa.float()-bb.float()).abs().max())
reload_decisions_equal=bool(torch.equal(aa.argmax(-1),bb.argmax(-1)))
reload_close=bool(torch.allclose(aa.float(),bb.float(),rtol=2e-3,atol=2e-3))
assert reload_decisions_equal and reload_close,(max_abs_diff,reload_close)
del reloaded;gc.collect();torch.cuda.empty_cache()
print("STRICT OFFLINE RELOAD PASS",max_abs_diff)
print("New model FP32 bytes:",model_bytes,
      "reference bytes:",baseline_bytes,
      "saved:",actual_saved_bytes)


STRICT OFFLINE RELOAD PASS 0.0
New model FP32 bytes: 1553602096 reference bytes: 1684141004 saved: 130538908


In [11]:
#@title 11. Publish only if CeNN improves gold; always write research report
val_delta_pp=100*(FINAL_VAL["gold_accuracy"]-
                   BASELINE_VAL["gold_accuracy"])
test_delta_vs_teacher=100*(test_quality["gold_accuracy"]-
                           test_quality["teacher_gold_accuracy"])
test_delta_vs_baseline=100*(test_quality["gold_accuracy"]-
                            baseline_test["gold_accuracy"])
speed_ok=(speed_against_baseline["min_gpu_speedup"]>=MIN_SPEED_RATIO and
          speed_against_baseline["min_wall_speedup"]>=MIN_SPEED_RATIO)
gates=dict(
 validation_gold_improvement=val_delta_pp>=MIN_VAL_GOLD_GAIN_PP,
 test_better_than_teacher=test_delta_vs_teacher>=MIN_TEST_GOLD_GAIN_PP,
 test_not_worse_than_baseline=test_delta_vs_baseline>=MIN_TEST_BASELINE_GAIN_PP,
 teacher_agreement=test_quality["teacher_agreement"]>=MIN_TEACHER_AGREEMENT,
 test_JS=test_quality["mean_js"]<=MAX_TEST_JS,
 saved_checkpoint_bytes=actual_saved_bytes>=MIN_CHECKPOINT_BYTES_SAVED,
 strict_offline_reload=reload_decisions_equal and reload_close,
 speed_nonregression=speed_ok)
gates["all_pass"]=(
    all(v for name,v in gates.items() if name!="speed_nonregression")
    and (speed_ok or not REQUIRE_SPEED_NONREGRESSION))
full_report=dict(
 experiment="V5.3 CeNN-inspired liquid FFN token-lattice recurrence",
 liquid_config=LIQUID_CONFIG,
 baseline_source=BASELINE_SOURCE,
 source_revision=PINNED_TEACHER_REVISION,
 selected_projection_count=len(SELECTED_SPECS),
 selected_ce_nn_layer=SELECTED_LAYER,
 compressed_projection_specs=SELECTED_SPECS,
 replacement_screen=scan,
 mode=SELECTED_MODE,
 selected_step=best_step,
 validation_student=FINAL_VAL,
 validation_baseline=BASELINE_VAL,
 validation_gain_pp=val_delta_pp,
 test_student=test_quality,
 test_baseline=baseline_test,
 gold_gain_vs_teacher_pp=test_delta_vs_teacher,
 gold_gain_vs_baseline_pp=test_delta_vs_baseline,
 test_error_breakdown=test_errors,
 paired_speed_vs_baseline=speed_against_baseline,
 paired_speed_vs_teacher=speed_against_teacher,
 original_fp32_checkpoint_bytes=baseline_bytes,
 liquid_fp32_checkpoint_bytes=model_bytes,
 fp32_storage_saving_bytes=actual_saved_bytes,
 reload_max_abs_diff=max_abs_diff,
 release_gates=gates,
 caveat="CeNN-inspired token-neighbor recurrence is not proven equivalent to Chua CeNN or an LTC solver. In residual mode the original FFN projection remains; no parameter compression is claimed for that layer. The test split may have been used in prior experiments.")
(DEST/"v53_report.json").write_text(json.dumps(full_report,indent=2))
card=f"""---
library_name: pytorch
base_model: {TEACHER_REPO}
tags:
- modernbert
- liquid-cenn
- recurrent-neural-network
- svd
- experimental
---

# Laya V5.3 — Liquid-CeNN FFN experiment

**CeNN-inspired 1D liquid token-neighbor recurrence**, mode
**{SELECTED_MODE}**, at encoder layer **{SELECTED_LAYER}**.
The 28-layer pretrained ModernBERT encoder and the original decision
head remain; 20 selected projection matrices still use low-rank
factorization / shared-delta approximation.

CeNN cell: `{CE_WIDTH}` state channels, `{CE_STEPS}` recurrent updates,
local depthwise 3-token kernel and input-dependent relaxation gate.

- Gold validation baseline: **{BASELINE_VAL['gold_accuracy']:.3%}**
- Gold validation CeNN: **{FINAL_VAL['gold_accuracy']:.3%}**
- Gold TEST teacher: **{test_quality['teacher_gold_accuracy']:.3%}**
- Gold TEST compressed baseline: **{baseline_test['gold_accuracy']:.3%}**
- Gold TEST liquid model: **{test_quality['gold_accuracy']:.3%}**
- TEST gold change vs baseline: **{test_delta_vs_baseline:+.2f}pp**
- Teacher decision agreement: **{test_quality['teacher_agreement']:.3%}**
- GPU median speed ratio vs baseline: **{speed_against_baseline['median_gpu_speedup']:.3f}×**
- FULL standalone FP32 weights: **{model_bytes:,} bytes**
- Reduction vs original FP32 weights: **{actual_saved_bytes:,} bytes**
- All release gates passed: **{gates['all_pass']}**

This model does **not** claim to have replaced standard attention with
a canonical CeNN. If mode is residual, the old FFN projection remains;
it is a liquid augmenting module, not a faster or smaller replacement.
All results are limited to the reported decision benchmark.

## Fully standalone load

```python
from modeling_laya_v53 import load_v53_model
model,tokenizer=load_v53_model("/path/to/model",device="cuda")
```
"""
(DEST/"README.md").write_text(card,encoding="utf-8")
(DEST/"requirements.txt").write_text(
    "torch>=2.4\ntransformers==4.57.6\nsafetensors>=0.4\nhuggingface_hub>=0.25,<1\n")
(DEST/"provenance.json").write_text(json.dumps(dict(
    source=TEACHER_REPO,revision=PINNED_TEACHER_REVISION,
    liquid_cell=LIQUID_CONFIG,selection=SELECTED_SPECS,
    baseline_source=BASELINE_SOURCE),indent=2))
HF_STATUS="blocked_by_release_gates"
HF_ERROR=None
if gates["all_pass"] and PUBLISH_TO_HF:
    try:
        notebook_login()
        uploader=HfApi()
        assert uploader.whoami()["name"]=="vtava"
        if uploader.repo_exists(STUDENT_REPO,repo_type="model"):
            existing=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
            if existing-{".gitattributes",".gitignore"} and not ALLOW_UPDATE_EXISTING:
                raise RuntimeError("Existing V5.3 repository has files; not overwriting")
        else:
            uploader.create_repo(repo_id=STUDENT_REPO,repo_type="model",
                                 private=PRIVATE_HF_REPO,exist_ok=False)
        uploader.upload_large_folder(repo_id=STUDENT_REPO,repo_type="model",
                                     folder_path=str(DEST))
        files_on_hf=set(uploader.list_repo_files(STUDENT_REPO,repo_type="model"))
        expected={"model.safetensors","modeling_laya_v53.py",
                  "config.json","v53_report.json","README.md",
                  "encoder/config.json"}
        assert expected.issubset(files_on_hf),expected-files_on_hf
        HF_STATUS="published"
        print("HF PUBLICATION VERIFIED:",STUDENT_REPO)
    except Exception as exc:
        HF_STATUS="upload_error"
        HF_ERROR=f"{type(exc).__name__}: {exc}"
        print("HF upload failed; archive will still download:",HF_ERROR)
elif gates["all_pass"]:
    HF_STATUS="disabled"
(OUT/"huggingface_status.json").write_text(json.dumps(dict(
    status=HF_STATUS,error=HF_ERROR,repo=STUDENT_REPO,
    gates=gates),indent=2))
print("V5.3 GATES:",json.dumps(gates,indent=2))


V5.3 GATES: {
  "validation_gold_improvement": false,
  "test_better_than_teacher": false,
  "test_not_worse_than_baseline": true,
  "teacher_agreement": true,
  "test_JS": true,
  "saved_checkpoint_bytes": true,
  "strict_offline_reload": true,
  "speed_nonregression": true,
  "all_pass": false
}


In [12]:
#@title 12. ZIP ALL V5.3 results, checkpoints, full model and AUTO-DOWNLOAD
# This cell is deliberately independent of the HF publication gates.
# It is safe to run again without retraining or uploading to HF.
import json,hashlib,zipfile
from pathlib import Path
from datetime import datetime,timezone

if "OUT" not in globals():
    OUT=Path("/content/Laya_V53_LiquidCeNN")
OUT=Path(OUT).resolve()
assert OUT.is_dir(),f"Run output directory missing: {OUT}"
MODEL_DIR=OUT/"standalone_model"
ZIP_PATH=OUT.parent/"Laya_V53_All_Results.zip"

# Explicit summary for reviewers; do not claim success when release gates fail.
summary={
    "experiment":"Laya V5.3 CeNN-inspired liquid FFN replacement/residual experiment",
    "generated_utc":datetime.now(timezone.utc).isoformat(),
    "notebook":"Laya_V53_Liquid_CeNN_Colab.ipynb",
    "results_dir":str(OUT),
    "model_directory_in_zip":"Laya_V53_LiquidCeNN/standalone_model/",
}
if "gates" in globals():
    summary["release_gates"]=gates
if "HF_STATUS" in globals():
    summary["hugging_face_status"]=HF_STATUS
if "HF_ERROR" in globals():
    summary["hugging_face_error"]=HF_ERROR
if "best_step" in globals():
    summary["selected_training_step"]=int(best_step)
if "FINAL_VAL" in globals():
    summary["validation"]=FINAL_VAL
if "test_quality" in globals():
    summary["independent_test"]=test_quality
if "test_delta_vs_teacher" in globals():
    summary["test_gold_gain_pp"]=float(test_delta_vs_teacher)
(OUT/"all_results_summary.json").write_text(json.dumps(
    summary,indent=2,default=str),encoding="utf-8")

# Collect every generated artifact under OUT, including all saved logs,
# the best trainable-parameter checkpoint, the entire standalone encoder,
# tokenizer, model.safetensors, model card, provenance and final report.
# Do NOT include the downloaded teacher or Hugging Face cache: they are not
# part of the new standalone artifact. The ZIP lives outside OUT.
MANIFEST_PATH=OUT/"archive_manifest.json"
files_to_zip=sorted(p for p in OUT.rglob("*")
                    if p.is_file() and not p.is_symlink()
                    and p.resolve()!=MANIFEST_PATH.resolve()
                    and p.resolve()!=ZIP_PATH.resolve())
assert files_to_zip,"No result files found"
names={str((Path(OUT.name)/p.relative_to(OUT)).as_posix())
       for p in files_to_zip}
# An incomplete run is still downloadable; record missing outputs explicitly.
required_paths={
    "standalone_model/model.safetensors",
    "standalone_model/config.json",
    "standalone_model/modeling_laya_v53.py",
    "standalone_model/v53_report.json",
    "liquid_training_progress.json",
    "liquid_training_summary.json",
    "liquid_test_report.json",
    "best_liquid_trainable.safetensors",
}
present={p.relative_to(OUT).as_posix() for p in files_to_zip}
missing=sorted(required_paths-present)
if missing:
    print("WARNING: run has incomplete outputs; archiving everything available.")
    print("Missing expected output files:",missing)

def sha256_file(source):
    digest=hashlib.sha256()
    with open(source,"rb") as reader:
        for chunk in iter(lambda:reader.read(8*1024*1024),b""):
            digest.update(chunk)
    return digest.hexdigest()

manifest={
    "archive_format":"laya-v52-results-v1",
    "created_utc":datetime.now(timezone.utc).isoformat(),
    "contains_complete_standalone_model":len(missing)==0,
    "missing_expected_files":missing,
    "files":[
        {"path":(Path(OUT.name)/p.relative_to(OUT)).as_posix(),
         "bytes":p.stat().st_size,"sha256":sha256_file(p)}
        for p in files_to_zip
    ]
}
MANIFEST_PATH.write_text(json.dumps(manifest,indent=2),encoding="utf-8")
all_files=sorted(files_to_zip+[MANIFEST_PATH])
print("Packing",len(all_files),"files; source bytes:",
      sum(p.stat().st_size for p in all_files))
print("The full FP32 model may make the ZIP exceed 1 GB.")

# Streaming ZIP64, with no model bytes loaded into RAM.
# Safetensors are stored rather than redundantly compressed for speed.
with zipfile.ZipFile(ZIP_PATH,mode="w",allowZip64=True,
                     compression=zipfile.ZIP_DEFLATED,compresslevel=1) as out_zip:
    for p in all_files:
        arcname=(Path(OUT.name)/p.relative_to(OUT)).as_posix()
        method=(zipfile.ZIP_STORED if p.suffix in
                {".safetensors",".pt",".pth",".zip",".gz"} else
                zipfile.ZIP_DEFLATED)
        out_zip.write(p,arcname=arcname,compress_type=method)
# Validate the ZIP entries and byte-size metadata before launching download.
with zipfile.ZipFile(ZIP_PATH,mode="r") as check:
    expected={(Path(OUT.name)/p.relative_to(OUT)).as_posix()
              for p in all_files}
    actual=set(check.namelist())
    assert actual==expected,("ZIP entry mismatch",sorted(expected-actual),
                             sorted(actual-expected))
    assert all(check.getinfo(name).file_size==
               (OUT/Path(name).relative_to(OUT.name)).stat().st_size
               for name in actual)
    assert check.getinfo(f"{OUT.name}/archive_manifest.json").file_size>0

print("PASS: Full results ZIP created and entries verified")
print("ZIP:",ZIP_PATH,"bytes:",ZIP_PATH.stat().st_size)
print("Result files:",len(all_files),"missing:",len(missing))

# Auto-download: no extra user action is needed when running in Colab.
# If a browser blocks a very large file, the ZIP remains in /content,
# and a direct link is also displayed.
try:
    from IPython.display import display,FileLink
    display(FileLink(str(ZIP_PATH)))
except ImportError:
    pass
try:
    from google.colab import files as colab_files
    colab_files.download(str(ZIP_PATH))
    print("Download initiated:",ZIP_PATH.name)
except Exception as download_error:
    print("Automatic browser download unavailable or blocked:",
          repr(download_error))
    print("The completed ZIP remains available at:",ZIP_PATH)


Packing 25 files; source bytes: 1558394107
The full FP32 model may make the ZIP exceed 1 GB.
PASS: Full results ZIP created and entries verified
ZIP: /content/Laya_V53_All_Results.zip bytes: 1555433200
Result files: 25 missing: 0


/content/Laya_V53_All_Results.zip

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

Download initiated: Laya_V53_All_Results.zip
